# Thích ứng miền đối nghịch cho nhận dạng rò rỉ ống PVC áp lực thấp bằng âm thanh — tập `tapdulieuMix_16k`

**Adversarial Domain Adaptation for Acoustic Leak Detection in Low-Pressure PVC Pipes across Acquisition Domains**

---

### Bối cảnh / Motivation

Pipeline v7 (`leak-cnn-pvc-kaggle-v7.ipynb`) huấn luyện và đánh giá **trong cùng một miền** thu âm.
Khi đem mô hình sang một miền khác (phòng thí nghiệm → hiện trường, cảm biến khác, điểm khảo sát khác,
chuỗi thu 8 kHz → 16 kHz...), phân bố đặc trưng **P(x)** thay đổi trong khi quan hệ vật lý
"có rò rỉ ↔ tín hiệu" **P(y|x)** được giả thiết giữ nguyên → **dịch chuyển hiệp biến (covariate shift)**.
Notebook này đo mức dịch chuyển đó, rồi áp dụng **thích ứng miền đối nghịch** (DANN, CDAN+E) để học
đặc trưng **phân biệt được rò rỉ nhưng không phân biệt được miền**, dùng **dữ liệu miền đích không nhãn**.

### Câu hỏi nghiên cứu / Research questions

| RQ | Câu hỏi | Bằng chứng trong notebook |
|---|---|---|
| **RQ1** | Các miền trong `tapdulieuMix_16k` khác nhau đến mức nào, và dấu hiệu rò rỉ có **cùng chiều** giữa các miền không? | Mục 3: băng thông, PSD, AUC theo dải trong từng miền, ma trận "huấn luyện miền A → kiểm thử miền B", PAD đầu vào |
| **RQ2** | Thích ứng đối nghịch có cải thiện nhận dạng **mức bản ghi ở miền đích** so với *source-only* và các phương pháp không đối nghịch (AdaBN, Deep CORAL) không? | Mục 9: bảng chính + ΔAUC ghép cặp, bootstrap theo bản ghi, Holm |
| **RQ3** | Thích ứng làm gì với không gian đặc trưng (căn chỉnh miền, giữ cấu trúc lớp)? | Mục 10–11: động học huấn luyện, PAD theo epoch, t-SNE, PAD/MMD theo lớp |
| **RQ4** | Kết quả nhạy thế nào với λ, augmentation, và **cách chọn mô hình khi không có nhãn đích**? | Mục 12–13: quy tắc chọn (last / source-val / DEV / oracle), quét λ, ablation |
| **RQ5** | Cần bao nhiêu bản ghi đích **có nhãn** để vượt UDA? (tuỳ chọn) | Mục 14: đường cong bán giám sát |

### Phương pháp so sánh / Compared methods

| Mã | Phương pháp | Dùng dữ liệu đích | Vai trò |
|---|---|---|---|
| `source_only` | Huấn luyện chỉ trên miền nguồn | không | **cận dưới** |
| `adabn` | Source-only + ước lượng lại BatchNorm trên đích (Li et al., 2018) | không nhãn | baseline không huấn luyện |
| `coral` | Deep CORAL — khớp hiệp phương sai (Sun & Saenko, 2016) | không nhãn | baseline không đối nghịch |
| `mmd` *(tuỳ chọn)* | DAN — MK-MMD (Long et al., 2015) | không nhãn | baseline không đối nghịch |
| **`dann`** | **DANN — Gradient Reversal Layer** (Ganin et al., 2016) | không nhãn | **phương pháp chính** |
| **`cdan_e`** | **CDAN+E — đối nghịch có điều kiện + trọng số entropy** (Long et al., 2018) | không nhãn | **biến thể đối nghịch có điều kiện** |
| `oracle` | Nguồn + đích **có nhãn** (giám sát đầy đủ) | có nhãn | **cận trên tham chiếu** |

### Giao thức đánh giá (tóm tắt)

1. **Đơn vị thống kê = bản ghi** (`record_uid = miền|Rxxx`; mã `Rxxx` trùng nhau giữa các miền được tách riêng).
2. **Quy nạp (inductive) theo K-fold trên bản ghi đích**: dữ liệu đích không nhãn dùng khi huấn luyện **không bao giờ**
   chứa bản ghi đang được kiểm thử → không có rò rỉ kiểu *transductive*.
3. **Không dùng nhãn đích để chọn mô hình**: quy tắc chính đăng ký trước là `last` (lịch huấn luyện cố định);
   `source-val`, `DEV` (You et al., 2019) là phân tích độ nhạy; `oracle` chỉ để **đo mức lạc quan**.
4. **CV lặp** (mặc định 3 seed: phân hoạch mới + khởi tạo mới) và **trung bình OOF qua các seed** làm ước lượng chính.
5. **CI bootstrap theo bản ghi**, **ΔAUC ghép cặp** so với *source-only*, **hiệu chỉnh Holm**.

> **Cách dùng nhanh / Quick start**
> 1. Tải thư mục `tapdulieuMix_16k` lên Kaggle (`Datasets → New dataset`), rồi `Add Data` vào notebook.
> 2. Cấu trúc khuyến nghị: `tapdulieuMix_16k/<miền>/<leak|noleak>/Rxxx_*.wav`
>    (hoặc `tapdulieuMix_16k/<leak|noleak>/<miền>/...`). Notebook **tự dò** miền từ thư mục; nếu miền nằm trong tên tệp
>    hay trong một CSV, chỉnh `cfg.domain_source` ở mục 1.2.
> 3. **Kiểm tra mục 2.3–2.4** (miền nào được nhận ra, mỗi miền bao nhiêu bản ghi mỗi lớp) trước khi để chạy hết.
> 4. `Settings → Accelerator = GPU T4 x2`, rồi `Run All`. Cấu hình mặc định ~1.5–3 giờ tuỳ kích thước dữ liệu.
>    Không tìm thấy dữ liệu → notebook tự sinh **dữ liệu mô phỏng hai miền** (SYNTHETIC DEMO MODE).
> 5. Chạy thử nhanh: biến môi trường `LEAK_SMOKE=1` (vài vòng lặp, 2 fold, 1 seed).

## 0. Cơ sở lý thuyết / Theoretical background

**Cận sai số miền đích (Ben-David et al., 2010).** Với mọi giả thuyết $h \in \mathcal{H}$:

$$\varepsilon_T(h) \;\le\; \varepsilon_S(h) \;+\; \tfrac{1}{2}\, d_{\mathcal{H}\Delta\mathcal{H}}(\mathcal{D}_S, \mathcal{D}_T) \;+\; \lambda^\*$$

Sai số đích bị chặn bởi sai số nguồn, **độ lệch giữa hai miền** đo bằng một bộ phân lớp miền, và sai số của
giả thuyết lý tưởng chung $\lambda^\*$. Độ lệch được ước lượng thực nghiệm bằng **Proxy A-distance**
$\mathrm{PAD} = 2(1 - 2\epsilon)$, với $\epsilon$ là sai số của bộ phân lớp nguồn-vs-đích (mục 3.7, 10, 11).

**DANN (Ganin et al., 2016).** Bộ trích đặc trưng $G_f$, bộ dự đoán nhãn $G_y$, bộ phân biệt miền $G_d$:

$$\min_{G_f, G_y}\; \max_{G_d}\;\; \mathcal{L}_y(G_y(G_f(x^s)), y^s) \;-\; \lambda_p\, \mathcal{L}_d\big(G_d(G_f(x)), d\big)$$

thực hiện bằng **lớp đảo gradient (GRL)**: truyền xuôi là ánh xạ đồng nhất, truyền ngược nhân gradient với $-\lambda_p$.
Hệ số tăng dần $\lambda_p = \lambda_{\max}\left(\frac{2}{1+e^{-\gamma p}} - 1\right)$, $\gamma = 10$, $p \in [0,1]$ là tiến độ huấn luyện.

**CDAN+E (Long et al., 2018).** $G_d$ nhận **ánh xạ đa tuyến** $f \otimes g$ (đặc trưng ⊗ xác suất lớp) để căn chỉnh
**phân bố có điều kiện theo lớp**, và mỗi mẫu được đánh trọng số $w = 1 + e^{-H(g)}$ để ưu tiên mẫu dự đoán chắc chắn.

**Giới hạn đã biết — phải kiểm tra trên dữ liệu thật:**

- **Dịch chuyển nhãn** (tỉ lệ leak/noleak khác nhau giữa miền): căn chỉnh biên P(f) có thể *làm hại*
  (Zhao et al., 2019). → mục 2.4 in tỉ lệ lớp theo miền; nguồn được lấy mẫu **cân bằng lớp**; CDAN+E giảm nhẹ vấn đề.
- **Dịch chuyển có điều kiện** (dấu hiệu rò rỉ đổi chiều giữa các miền): khi đó không phương pháp UDA nào đảm bảo
  cải thiện. → mục 3.5 kiểm tra chiều của AUC theo dải trong từng miền.
- **Chọn mô hình không có nhãn đích**: chọn epoch/siêu tham số bằng nhãn đích làm kết quả UDA **lạc quan giả tạo**
  (You et al., 2019). → mục 12 đo trực tiếp mức lạc quan này.

**Tài liệu tham khảo chính.** Ben-David et al., *Mach. Learn.* 79 (2010) · Ganin et al., *JMLR* 17 (2016) ·
Long et al., *NeurIPS* (2018) · Sun & Saenko, *ECCV-W* (2016) · Long et al., *ICML* (2015) ·
Li et al., *Pattern Recognit.* 80 (2018) · Zhao et al., *ICML* (2019) · You et al., *ICML* (2019).

## 1. Thiết lập môi trường / Environment setup

In [ ]:
# =========================================================
# 1.1 Imports & phụ thuộc / Imports and dependencies
# =========================================================
import os, sys, gc, json, math, re, time, random, shutil, hashlib, warnings, subprocess
import threading, pickle, copy, queue, traceback, zipfile, io
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict, field
from collections import defaultdict

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

def _pip_install(pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
    if r.returncode != 0:
        print(f"  !! pip install thất bại cho {pkgs} (mã {r.returncode}).")
    return r.returncode == 0

try:
    import soundfile as sf
except Exception:
    _pip_install(["soundfile"]); import soundfile as sf
try:
    import librosa
except Exception:
    _pip_install(["librosa"]); import librosa

import numpy as np
import pandas as pd
import scipy.signal as sps
import scipy.stats as spstats

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, Markdown, Image as IPyImage

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_fscore_support,
                             confusion_matrix, matthews_corrcoef, balanced_accuracy_score,
                             accuracy_score, silhouette_score)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def amp_autocast(device, enabled):
    try:
        return torch.amp.autocast(device_type=torch.device(device).type, enabled=bool(enabled))
    except Exception:
        return torch.cuda.amp.autocast(enabled=bool(enabled))

def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=bool(enabled))
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=bool(enabled))

T_START = time.time()
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__, "| pandas:", pd.__version__)
print("PyTorch     :", torch.__version__, "| torchaudio:", torchaudio.__version__)
print("librosa     :", librosa.__version__)
print("CUDA        :", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {p.name} | {p.total_memory/1024**3:.1f} GB")

In [ ]:
# =========================================================
# 1.2 Cấu hình trung tâm / Central configuration
#     Mọi tham số thí nghiệm nằm ở đây. Ghi chú [P] = tham số ĐĂNG KÝ TRƯỚC (không chỉnh theo
#     kết quả trên miền đích); [S] = chỉ dùng cho phân tích độ nhạy.
# =========================================================
@dataclass
class CFG:
    # ---- Dữ liệu / Data ------------------------------------------------------
    data_root: str = ""                       # để trống -> tự dò trong /kaggle/input theo dataset_hint
    dataset_hint: str = "tapdulieuMix_16k"     # tên thư mục gốc của tập Mix
    leak_keys: tuple = ("leak", "ro_ri", "rori")
    noleak_keys: tuple = ("noleak", "no_leak", "non_leak", "nonleak", "normal",
                          "background", "khong_ro_ri", "khongroli", "no-leak")
    audio_ext: tuple = (".wav", ".flac", ".ogg", ".mp3", ".WAV")
    record_regex: str = r"(R\d{1,4})"          # mã bản ghi trong tên tệp
    strict_checks: bool = True                 # dừng khi có mâu thuẫn nhãn

    # ---- MIỀN / Domains ------------------------------------------------------
    # auto  : thư mục (không phải leak/noleak) gần gốc nhất trên đường dẫn tương đối = miền;
    #         nếu không có, thử domain_regex trên đường dẫn.
    # folder: chỉ dùng thư mục | regex: chỉ dùng domain_regex | csv: tệp 2 cột record_id,domain
    domain_source: str = "auto"
    domain_depth: int = 1                      # số cấp thư mục ghép thành tên miền
    domain_regex: str = ""                     # ví dụ r"^(lab|field)_" trên tên tệp/đường dẫn
    domain_regex_group: int = 1
    domain_csv: str = ""
    domain_alias: dict = field(default_factory=dict)   # gộp/đổi tên miền, ví dụ {"ptn": "lab"}
    exclude_domains: tuple = ()
    min_records_per_class: int = 2             # miền cần >= ngần này bản ghi MỖI lớp để làm đích

    # ---- Bài toán chuyển miền / Transfer tasks -------------------------------
    # () = tự động: 2 miền -> A→B và B→A; >2 miền -> leave-one-domain-out (các miền còn lại → miền giữ lại)
    # Khai báo tay: (("lab", "field"), ("lab+field8k", "field16k"))
    tasks: tuple = ()
    max_tasks: int = 6

    # ---- Tín hiệu / Signal ---------------------------------------------------
    target_sr: int = 16000
    highpass_hz: float = 20.0
    notch_mains: bool = True
    mains_hz: float = 50.0
    mains_harmonics: int = 6
    notch_q: float = 30.0
    frame_sec: float = 2.0
    hop_sec: float = 1.0
    # [P] Hoà hợp băng thông: nếu một miền có băng thông hiệu dụng thấp (ví dụ thu 8 kHz rồi nâng
    #     lên 16 kHz), mọi front-end phổ bị giới hạn về băng thông CHUNG -> loại một dấu hiệu "miền"
    #     tầm thường (năng lượng > 4 kHz). "auto" | "off" | số Hz.
    bandwidth_harmonize: str = "auto"
    bw_energy_frac: float = 0.995
    check_duplicates: bool = True              # dò bản ghi trùng nội dung giữa/trong các miền

    # ---- Biểu diễn / Front-end ----------------------------------------------
    n_fft: int = 2048                          # 128 ms @ 16 kHz
    hop_length: int = 512                      # 32 ms
    n_mels: int = 64
    f_min: float = 20.0
    f_max: float = 8000.0
    specaug_freq_mask: int = 8
    specaug_time_mask: int = 8

    # ---- Kiến trúc / Architecture -------------------------------------------
    backbone: str = "cnn2d"                    # cnn2d (log-Mel, như LeakCNN2D v7) | cnn1d (dạng sóng)
    widths: tuple = (32, 64, 128, 256)
    bottleneck_dim: int = 256
    disc_hidden: int = 256
    disc_dropout: float = 0.5

    # ---- Phương pháp / Methods ----------------------------------------------
    methods: tuple = ("source_only", "adabn", "coral", "dann", "cdan_e", "oracle")
    lambda_max: float = 1.0                    # [P] hệ số GRL cuối lịch (DANN/CDAN)
    lambda_gamma: float = 10.0
    coral_weight: float = 1.0
    mmd_weight: float = 1.0

    # ---- Giao thức / Protocol -----------------------------------------------
    target_protocol: str = "inductive"         # [P] inductive (K-fold bản ghi đích) | transductive
    tgt_folds: int = 5
    src_val_frac: float = 0.2                  # bản ghi nguồn giữ lại (chọn mô hình, ngưỡng, DEV)
    seeds: tuple = (42, 43, 44)                # CV lặp: mỗi seed = phân hoạch mới + khởi tạo mới

    # ---- Huấn luyện / Training ----------------------------------------------
    epochs: int = 30
    iters_per_epoch: int = 80                  # "epoch" = số vòng lặp cố định (chuẩn của DANN)
    batch_size: int = 64                       # MỖI miền trong một vòng lặp
    lr: float = 3e-4
    disc_lr_mult: float = 1.0
    weight_decay: float = 1e-4
    warmup_frac: float = 0.05
    label_smoothing: float = 0.05
    grad_clip: float = 5.0
    amp: bool = True
    class_balanced_source: bool = True         # lô nguồn cân bằng lớp (giảm ảnh hưởng dịch chuyển nhãn)
    augment: bool = True
    aug_noise_p: float = 0.5
    aug_noise_snr_db: tuple = (10.0, 30.0)
    aug_fir_p: float = 0.5                     # đáp ứng kênh ngẫu nhiên (bất biến cảm biến)
    aug_fir_taps: int = 48
    aug_shift_p: float = 0.5
    eval_every: int = 2                        # đánh giá mỗi ngần này epoch
    select_rule: str = "last"                  # [P] last | src_val | dev   (oracle chỉ để đo lạc quan)
    select_from_frac: float = 0.3              # không chọn checkpoint trước 30% lịch (λ còn nhỏ)
    dev_max: int = 1500                        # số frame tối đa mỗi phía cho DEV/PAD

    # ---- Thống kê / Statistics ----------------------------------------------
    record_agg: str = "p_mean"                 # [P] xác suất bản ghi = trung bình xác suất frame
    threshold: float = 0.5                     # [P] ngưỡng quyết định mức bản ghi
    bootstrap_n: int = 2000

    # ---- Thí nghiệm bổ sung / Extra experiments -----------------------------
    run_lambda_sweep: bool = True
    lambda_grid: tuple = (0.1, 0.3, 3.0)       # cộng với lambda_max của lần chạy chính
    run_ablation: bool = True
    ablation_variants: tuple = ("no_fir", "no_aug", "no_balance", "cdan_noent")
    sweep_seeds: tuple = (42,)
    sweep_task: int = 0
    run_ssda: bool = False                     # đường cong bán giám sát (tốn thêm thời gian)
    ssda_shots: tuple = (1, 2, 4)              # số bản ghi đích có nhãn MỖI lớp
    run_band_occlusion: bool = True
    tsne_max_points: int = 3000

    # ---- Phần cứng & đầu ra / Hardware and outputs --------------------------
    parallel_jobs: bool = True                 # mỗi GPU chạy một job (không dùng DataParallel)
    max_parallel: int = 2
    gpu_resident_max_gb: float = 6.0
    out_dir: str = "/kaggle/working"
    fig_dpi: int = 300
    save_models: bool = True
    resume: bool = True                        # dùng lại kết quả job đã chạy (cùng cấu hình)

    smoke_test: bool = False


cfg = CFG()
if os.environ.get("LEAK_SMOKE") == "1":
    cfg.smoke_test = True
if os.environ.get("LEAK_DATA") is not None:
    cfg.data_root = os.environ["LEAK_DATA"]
if os.environ.get("LEAK_OUT"):
    cfg.out_dir = os.environ["LEAK_OUT"]
elif not Path("/kaggle").exists():
    cfg.out_dir = str(Path("./ada_outputs").resolve())

if cfg.smoke_test:
    print("!! SMOKE TEST: vài vòng lặp, 2 fold, 1 seed — chỉ để kiểm tra toàn tuyến.")
    cfg.epochs, cfg.iters_per_epoch, cfg.batch_size, cfg.eval_every = 4, 4, 16, 1
    cfg.tgt_folds, cfg.seeds, cfg.bootstrap_n = 2, (42,), 100
    cfg.methods = ("source_only", "adabn", "coral", "mmd", "dann", "cdan_e", "oracle")
    cfg.lambda_grid, cfg.ablation_variants = (0.3,), ("no_fir", "cdan_noent")
    cfg.run_ssda, cfg.ssda_shots = True, (1,)
    cfg.tsne_max_points, cfg.dev_max = 300, 200

if not torch.cuda.is_available():
    cfg.amp = False
    cfg.parallel_jobs = False

OUT = Path(cfg.out_dir)
FIG_DIR = OUT / "figures";      FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = OUT / "tables";       TAB_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT / "cache";      CACHE_DIR.mkdir(parents=True, exist_ok=True)

FRAME_LEN = int(round(cfg.frame_sec * cfg.target_sr))
HOP_LEN = int(round(cfg.hop_sec * cfg.target_sr))
TOTAL_ITERS = int(cfg.epochs * cfg.iters_per_epoch)
SELECT_FROM_EPOCH = int(math.ceil(cfg.select_from_frac * cfg.epochs))
assert cfg.select_rule in ("last", "src_val", "dev"), "select_rule chính phải KHÔNG dùng nhãn đích"
assert cfg.epochs > SELECT_FROM_EPOCH

_cfg_print = {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()}
print(json.dumps(_cfg_print, indent=1, ensure_ascii=False))
print(f"\nFrame {cfg.frame_sec}s = {FRAME_LEN} mẫu | hop {cfg.hop_sec}s | "
      f"{cfg.epochs} epoch x {cfg.iters_per_epoch} vòng = {TOTAL_ITERS} vòng lặp/job")
print(f"Chọn checkpoint chính: '{cfg.select_rule}' (chỉ từ epoch {SELECT_FROM_EPOCH}) | "
      f"giao thức đích: {cfg.target_protocol} | seeds {list(cfg.seeds)}")
print("Thư mục ra:", OUT)

In [ ]:
# =========================================================
# 1.3 Tái lập, phong cách hình vẽ, xuất bảng / Reproducibility, plot style, table export
# =========================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    return int(seed)

set_seed(cfg.seeds[0])

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": cfg.fig_dpi, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": ":",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.facecolor": "white",
    "figure.constrained_layout.use": True,
})
PAL = {"leak": "#C8553D", "noleak": "#2E6F95", "grey": "#6E7B8B", "warn": "#E0A458",
       "src": "#2E6F95", "tgt": "#C8553D", "ok": "#3F7D20"}
DOMAIN_PALETTE = ["#2E6F95", "#C8553D", "#3F7D20", "#8C6BB1", "#E0A458", "#4C9F9F", "#A0522D"]
METHOD_LABEL = {"source_only": "Source-only", "adabn": "AdaBN", "coral": "Deep CORAL",
                "mmd": "DAN (MK-MMD)", "dann": "DANN", "cdan": "CDAN", "cdan_e": "CDAN+E",
                "oracle": "Oracle (S+T labelled)", "oracle_shots": "Source + few target labels",
                "dann_shots": "DANN + few target labels"}
METHOD_COLOR = {"source_only": "#6E7B8B", "adabn": "#4C9F9F", "coral": "#8C6BB1", "mmd": "#A0522D",
                "dann": "#2E6F95", "cdan": "#E0A458", "cdan_e": "#C8553D", "oracle": "#3F7D20",
                "oracle_shots": "#3F7D20", "dann_shots": "#2E6F95"}
ADAPT_METHODS = ("coral", "mmd", "dann", "cdan", "cdan_e")      # dùng đích không nhãn khi huấn luyện
ADV_METHODS = ("dann", "cdan", "cdan_e")                        # có bộ phân biệt miền + GRL

def mlabel(m):
    return METHOD_LABEL.get(m, m)

FIG_INDEX, TAB_INDEX = {}, {}

def savefig(fig, name, caption=""):
    # Lưu PNG (xem nhanh) + PDF (vector cho manuscript) và HIỂN THỊ ngay trong notebook.
    n = len(FIG_INDEX) + 1
    stem = f"Fig{n:02d}_{name}"
    fig.savefig(FIG_DIR / f"{stem}.png")
    fig.savefig(FIG_DIR / f"{stem}.pdf")
    FIG_INDEX[stem] = caption
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110)      # bản xem trước, không phụ thuộc backend
    display(IPyImage(data=buf.getvalue()))
    plt.close(fig)
    if caption:
        display(Markdown(f"**Figure {n}.** {caption}"))
    return stem

def _fmt(v, nd=4):
    if isinstance(v, (float, np.floating)):
        return "–" if not np.isfinite(v) else f"{v:.{nd}f}"
    return str(v)

def _tex_escape(s):
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"), ("→", r"$\rightarrow$"),
                 ("±", r"$\pm$"), ("Δ", r"$\Delta$"), ("λ", r"$\lambda$")]:
        s = s.replace(a, b)
    return s

def df_to_latex(df, caption, label, nd=4):
    cols = list(df.columns)
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             r"\caption{" + _tex_escape(caption) + "}", r"\label{" + label + "}",
             r"\begin{tabular}{" + "l" * len(cols) + "}", r"\toprule",
             " & ".join(_tex_escape(str(c)) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(_tex_escape(_fmt(r[c], nd)) for c in cols) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

def df_to_markdown(df, nd=4):
    cols = list(df.columns)
    out = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(_fmt(r[c], nd) for c in cols) + " |")
    return "\n".join(out)

def save_table(df, name, caption="", nd=4):
    # CSV (số liệu gốc) + LaTeX booktabs + Markdown, đánh số tự động như hình.
    n = len(TAB_INDEX) + 1
    stem = f"Table{n:02d}_{name}"
    df = df.reset_index(drop=True)
    df.to_csv(TAB_DIR / f"{stem}.csv", index=False)
    (TAB_DIR / f"{stem}.tex").write_text(df_to_latex(df, caption, f"tab:{name}", nd), encoding="utf-8")
    (TAB_DIR / f"{stem}.md").write_text(df_to_markdown(df, nd), encoding="utf-8")
    TAB_INDEX[stem] = caption
    display(Markdown(f"**Table {n}.** {caption}"))
    with pd.option_context("display.max_columns", 50, "display.width", 250):
        display(df.round(nd) if hasattr(df, "round") else df)
    return stem

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Đọc dữ liệu và nhận diện miền / Data discovery and domain inventory

Ba định danh được trích cho **mỗi tệp**:

| Định danh | Nguồn | Vai trò |
|---|---|---|
| `label` | thư mục `leak/` / `noleak/` (ưu tiên), rồi tên tệp | nhãn mức bản ghi |
| `domain` | thư mục không phải nhãn trên đường dẫn tương đối (hoặc regex / CSV) | miền thu âm |
| `record_uid = domain\|Rxxx` | mã `Rxxx` trong tên tệp **ghép với miền** | **đơn vị nhóm và đơn vị thống kê** |

> Ghép miền vào mã bản ghi là bắt buộc: tập *Mix* có thể chứa `R001` ở cả hai miền nhưng là hai lần thu khác nhau.
> Ngược lại, nếu **cùng một lần thu** xuất hiện ở hai miền (ví dụ bản 8 kHz và bản 16 kHz), mục 3.6 sẽ phát hiện
> nhờ so khớp đường bao năng lượng — đây là rò rỉ dữ liệu nguồn↔đích phải loại bỏ trước khi báo cáo.

In [ ]:
# =========================================================
# 2.1 Gán nhãn theo từ + dò thư mục dữ liệu / Word-level labelling and dataset discovery
# =========================================================
def _norm_token(s):
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower().replace("-", "_").replace(" ", "_")

def _key_regex(keys):
    ks = sorted({_norm_token(k) for k in keys}, key=len, reverse=True)
    return re.compile(r"(?:^|[_.])(?:" + "|".join(map(re.escape, ks)) + r")(?=$|[_.\d])")

_NOLEAK_RE = _key_regex(cfg.noleak_keys)
_LEAK_RE = _key_regex(cfg.leak_keys)

def _label_of_token(s):
    s = _norm_token(s)
    if _NOLEAK_RE.search(s):
        return 0
    if _LEAK_RE.search(s):
        return 1
    return None

def label_from_rel(rel):
    # Thư mục chứa tệp -> các thư mục cha (gần trước) -> tên tệp
    parts = list(Path(rel).parts)
    for tok in list(reversed(parts[:-1])) + [parts[-1]]:
        lab = _label_of_token(tok)
        if lab is not None:
            return lab, tok
    return None, None

for _t, _w in [("leak", 1), ("noleak", 0), ("No-Leak", 0), ("R001_leak01", 1), ("R12_noleak_03", 0),
               ("rò_rỉ", 1), ("khong_ro_ri", 0), ("leakage", None), ("tapdulieuMix_16k", None),
               ("field", None), ("lab", None)]:
    assert _label_of_token(_t) == _w, f"bộ gán nhãn sai với '{_t}'"

def _audio_in(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix in cfg.audio_ext)

def _has_both(root, files):
    labs = {label_from_rel(p.relative_to(root))[0] for p in files}
    return {0, 1}.issubset(labs)

def discover_root(explicit="", hint=""):
    if explicit:
        ex = Path(explicit)
        if ex.exists():
            fs = _audio_in(ex)
            if fs and _has_both(ex, fs):
                return ex, fs
            print(f"!! cfg.data_root = {ex} tồn tại nhưng không chứa đủ hai lớp leak/noleak.")
        else:
            print(f"!! cfg.data_root = {ex} KHÔNG tồn tại -> tự dò.")
    bases = [b for b in (Path("/kaggle/input"), Path("./data")) if b.exists()]
    # (1) thư mục trùng tên gợi ý (bỏ qua hoa/thường và ký tự không phải chữ-số: Kaggle đổi
    #     "tapdulieuMix_16k" thành slug "tapdulieumix-16k")
    _key = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
    if hint:
        for base in bases:
            for d in sorted(base.rglob("*")):
                if d.is_dir() and _key(d.name) == _key(hint):
                    fs = _audio_in(d)
                    if fs and _has_both(d, fs):
                        return d, fs
    # (2) tổ tiên chung thấp nhất của mọi tệp âm thanh có nhãn
    for base in bases:
        fs = [p for p in _audio_in(base) if label_from_rel(p.relative_to(base))[0] is not None]
        if fs and _has_both(base, fs):
            root = Path(os.path.commonpath([str(p.parent) for p in fs]))
            while _label_of_token(root.name) is not None:      # không lấy chính thư mục leak/
                root = root.parent
            return root, _audio_in(root)
    return None, []

DATA_ROOT, AUDIO_FILES = discover_root(cfg.data_root, cfg.dataset_hint)
SYNTHETIC = DATA_ROOT is None
if SYNTHETIC:
    print("!! KHÔNG tìm thấy dataset -> SYNTHETIC DEMO MODE (dữ liệu mô phỏng hai miền).")
else:
    print("Dataset root:", DATA_ROOT, "|", len(AUDIO_FILES), "tệp âm thanh")

def print_tree(root, max_depth=3, max_children=12):
    # Cây thư mục kèm số tệp âm thanh -> giúp cấu hình cfg.domain_source khi tự dò sai.
    root = Path(root)
    def _walk(d, depth):
        kids = sorted(k for k in d.iterdir() if k.is_dir())
        for k in kids[:max_children]:
            n = len(_audio_in(k))
            print("  " * depth + f"├─ {k.name}/  ({n} tệp)")
            if depth + 1 < max_depth:
                _walk(k, depth + 1)
        if len(kids) > max_children:
            print("  " * depth + f"└─ ... (+{len(kids) - max_children} thư mục)")
    print(f"{root.name}/  ({len(_audio_in(root))} tệp)")
    _walk(root, 0)

In [ ]:
# =========================================================
# 2.2 Dữ liệu mô phỏng HAI MIỀN / Two-domain synthetic data (chỉ khi không có dữ liệu thật)
#     "lab"  : nền nhiễu hồng, rò rỉ băng hẹp 350-900 Hz SNR cao, băng thông đầy đủ 8 kHz.
#     "field": nền nhiễu nâu + xe cộ + bơm, rò rỉ yếu hơn & dịch xuống 250-700 Hz, cảm biến có
#              độ dốc phổ khác, băng thông 3.8 kHz (mô phỏng chuỗi thu 8 kHz nâng lên 16 kHz),
#              tỉ lệ lớp khác "lab" (dịch chuyển nhãn), mã Rxxx TRÙNG với "lab".
# =========================================================
def _synthesise_mixed(root, spec, files_per_rec=2, dur=24.0, sr=16000, seed=0):
    rng = np.random.default_rng(seed)
    root = Path(root)
    if root.exists():
        try:
            shutil.rmtree(root)
        except Exception:
            root = Path(str(root) + "_" + time.strftime("%H%M%S"))
    n = int(dur * sr); t = np.arange(n) / sr
    f = np.fft.rfftfreq(n, 1 / sr); f[0] = f[1]

    def colored(alpha):
        X = np.fft.rfft(rng.standard_normal(n))
        return np.fft.irfft(X / f ** (alpha / 2), n)

    prof = {"lab": dict(alpha=1.0, snr=(-2, 8), f0=(350, 900), bw=(80, 200), hum=0.02,
                        tr=(0, 2), pump=0.0, lp=None, tilt=0.0),
            "field": dict(alpha=1.6, snr=(-8, 2), f0=(250, 700), bw=(60, 160), hum=0.08,
                          tr=(3, 7), pump=0.5, lp=3800.0, tilt=0.6)}
    for dom, n_leak, n_no in spec:
        P = prof[dom]; rid = 0
        for cls, n_r in (("leak", n_leak), ("noleak", n_no)):
            (root / dom / cls).mkdir(parents=True, exist_ok=True)
            for _ in range(n_r):
                rid += 1
                f0 = rng.uniform(*P["f0"]); bw = rng.uniform(*P["bw"])
                for seg in range(1, files_per_rec + 1):
                    x = colored(P["alpha"]); x /= np.std(x) + 1e-12
                    for h in range(1, 4):
                        x += P["hum"] * np.sin(2 * np.pi * 50 * h * t + rng.uniform(0, 6.28))
                    if cls == "leak":
                        b, a = sps.butter(4, [max(20., f0 - bw) / (sr / 2), (f0 + bw) / (sr / 2)], "band")
                        lk = sps.lfilter(b, a, rng.standard_normal(n)); lk /= np.std(lk) + 1e-12
                        am = 1 + 0.3 * np.sin(2 * np.pi * rng.uniform(.05, .3) * t)
                        x = x + lk * am * 10 ** (rng.uniform(*P["snr"]) / 20)
                    elif rng.random() < P["pump"]:                       # tiếng bơm: nhầm lẫn dễ có
                        fp = rng.uniform(100, 300)
                        x = x + 0.3 * np.sin(2 * np.pi * fp * t) * (1 + 0.2 * rng.standard_normal(n))
                    for _ in range(int(rng.integers(*P["tr"])) if P["tr"][1] > 0 else 0):
                        s0 = int(rng.integers(0, n - sr))
                        x[s0:s0 + sr] += 3.0 * np.exp(-np.arange(sr) / (sr * .05)) * rng.standard_normal(sr)
                    if P["tilt"]:
                        x = sps.lfilter([1 - P["tilt"]], [1, -P["tilt"]], x)
                    if P["lp"]:
                        b, a = sps.butter(8, P["lp"] / (sr / 2)); x = sps.filtfilt(b, a, x)
                    x = x / (np.max(np.abs(x)) + 1e-9) * 0.9
                    sf.write(root / dom / cls / f"R{rid:03d}_{cls}{seg:02d}.wav", x.astype(np.float32), sr)
    return root

if SYNTHETIC:
    if cfg.smoke_test:
        _spec, _fpr, _dur = (("lab", 5, 4), ("field", 4, 5)), 1, 12.0
    else:
        _spec, _fpr, _dur = (("lab", 14, 10), ("field", 10, 14)), 2, 24.0
    DATA_ROOT = _synthesise_mixed(OUT / "synthetic_mix_16k", _spec, _fpr, _dur, cfg.target_sr)
    AUDIO_FILES = _audio_in(DATA_ROOT)
    print("Synthetic dataset:", DATA_ROOT, "|", len(AUDIO_FILES), "tệp")

print_tree(DATA_ROOT)

In [ ]:
# =========================================================
# 2.3 Bảng kiểm kê mức TỆP: nhãn, MIỀN, mã bản ghi / File-level manifest
# =========================================================
_REC_RE = re.compile(cfg.record_regex, re.IGNORECASE)
DOMAIN_CSV_MAP = {}
if cfg.domain_source == "csv":
    _dm = pd.read_csv(cfg.domain_csv)
    assert {"record_id", "domain"}.issubset(_dm.columns), "CSV miền cần 2 cột: record_id, domain"
    DOMAIN_CSV_MAP = {str(a).strip().upper(): str(b).strip() for a, b in zip(_dm.record_id, _dm.domain)}

def extract_record_id(path):
    m = _REC_RE.search(path.stem)
    if m:
        return m.group(1).upper(), "regex:stem"
    m = _REC_RE.search(str(path.parent))
    if m:
        return m.group(1).upper(), "regex:parent"
    return path.stem, "fallback:stem"

def _domain_by_regex(rel):
    if not cfg.domain_regex:
        return None
    m = re.search(cfg.domain_regex, str(rel), re.IGNORECASE)
    return m.group(cfg.domain_regex_group) if m else None

def _domain_dirs(rel):
    return [d for d in Path(rel).parts[:-1] if _label_of_token(d) is None]

# Bỏ các thư mục "bọc" chung cho MỌI tệp (ví dụ tapdulieuMix_16k/data/<miền>/...) trước khi lấy miền
_all_dirs = [_domain_dirs(p.relative_to(DATA_ROOT)) for p in AUDIO_FILES
             if label_from_rel(p.relative_to(DATA_ROOT))[0] is not None]
WRAP_DEPTH = 0
while _all_dirs and all(len(d) > WRAP_DEPTH + 1 for d in _all_dirs) and \
        len({d[WRAP_DEPTH] for d in _all_dirs}) == 1:
    WRAP_DEPTH += 1
if WRAP_DEPTH:
    print(f"Bỏ {WRAP_DEPTH} cấp thư mục chung trước khi đọc tên miền: {'/'.join(_all_dirs[0][:WRAP_DEPTH])}/")

def extract_domain(rel, rid):
    if cfg.domain_source == "csv":
        return DOMAIN_CSV_MAP.get(str(rid).upper()), "csv"
    if cfg.domain_source == "regex":
        return _domain_by_regex(rel), "regex"
    dirs = _domain_dirs(rel)[WRAP_DEPTH:]
    if dirs:
        return "/".join(dirs[:cfg.domain_depth]), "folder"
    if cfg.domain_source == "auto":
        d = _domain_by_regex(rel)
        if d:
            return d, "regex"
    return "all", "none"

rows = []
for p_ in AUDIO_FILES:
    rel = p_.relative_to(DATA_ROOT)
    lab, tok = label_from_rel(rel)
    if lab is None:
        continue
    try:
        info = sf.info(str(p_))
    except Exception as e:
        print("  [bỏ qua]", rel, type(e).__name__); continue
    rid, how = extract_record_id(p_)
    dom, dsrc = extract_domain(rel, rid)
    if dom is None:
        print("  [không xác định miền]", rel); continue
    dom = cfg.domain_alias.get(dom, dom)
    rows.append(dict(path=str(p_), rel=str(rel), file_stem=p_.stem, domain=dom, domain_source=dsrc,
                     record_id=rid, rid_source=how, record_uid=f"{dom}|{rid}",
                     label=int(lab), cls="leak" if lab == 1 else "noleak", label_token=tok,
                     duration_s=float(info.duration), sr_native=int(info.samplerate),
                     channels=int(info.channels)))

files = pd.DataFrame(rows)
assert len(files), "Không đọc được tệp âm thanh nào."
if cfg.exclude_domains:
    files = files[~files.domain.isin(cfg.exclude_domains)].reset_index(drop=True)
files = files.sort_values(["domain", "cls", "record_uid", "file_stem"]).reset_index(drop=True)
files["file_id"] = np.arange(len(files))
files.to_csv(OUT / "manifest_files.csv", index=False)

print(f"Số tệp: {len(files)} | số miền: {files.domain.nunique()} -> {sorted(files.domain.unique())}")
print("Nguồn xác định miền:", files.domain_source.value_counts().to_dict())
print("Trích mã bản ghi   :", files.rid_source.value_counts().to_dict())
print("Tần số lấy mẫu gốc :", files.groupby("domain").sr_native.agg(lambda s: sorted(set(s))).to_dict())

if files.domain.nunique() < 2:
    print("\n!! CHỈ NHẬN RA MỘT MIỀN. Thích ứng miền cần ít nhất 2 miền. Cây thư mục ở trên cho thấy")
    print("   cấu trúc thực tế; hãy đặt cfg.domain_source='regex' + cfg.domain_regex (miền nằm trong tên")
    print("   tệp), hoặc 'csv' + cfg.domain_csv (bảng record_id,domain), rồi chạy lại từ mục 2.3.")
    raise ValueError("Cần >= 2 miền cho thích ứng miền (xem hướng dẫn ngay trên).")
files.head(6)

In [ ]:
# =========================================================
# 2.3b Kiểm tra chất lượng định danh / Identifier quality checks
# =========================================================
problems = []
conf = files.groupby("record_uid").label.nunique()
if (conf > 1).any():
    bad = conf[conf > 1].index.tolist()
    problems.append(f"{len(bad)} bản ghi mang hai nhãn")
    display(files[files.record_uid.isin(bad)][["record_uid", "cls", "rel"]])

n_fb = int((files.rid_source == "fallback:stem").sum())
if n_fb:
    print(f"!! {n_fb} tệp không khớp regex '{cfg.record_regex}' -> lấy tên tệp làm mã bản ghi "
          f"(mỗi tệp thành một bản ghi; kiểm tra lại nếu một lần thu bị cắt thành nhiều tệp).")

# Mã Rxxx trùng giữa các miền: hợp lệ (đã tách bằng record_uid) nhưng phải nêu ra.
_shared = files.groupby("record_id").domain.nunique()
_shared = _shared[_shared > 1]
if len(_shared):
    print(f"Mã bản ghi xuất hiện ở NHIỀU miền: {len(_shared)} (ví dụ {list(_shared.index[:6])}).")
    print("  -> được coi là các lần thu KHÁC NHAU (record_uid = miền|Rxxx). Mục 3.6 kiểm tra bằng nội")
    print("     dung âm thanh xem có phải cùng một lần thu bị sao chép sang miền khác hay không.")

fpr_ = files.groupby("record_uid").size()
print(f"Tệp/bản ghi: trung vị {int(fpr_.median())}, min {fpr_.min()}, max {fpr_.max()} | "
      f"bản ghi nhiều tệp: {(fpr_ > 1).sum()}/{len(fpr_)}")
if problems:
    msg = "CẦN XỬ LÝ: " + "; ".join(problems)
    print("\n=== " + msg + " ===")
    if cfg.strict_checks:
        raise ValueError(msg)
else:
    print("✓ Nhãn nhất quán trong mọi bản ghi.")

In [ ]:
# =========================================================
# 2.4 Kiểm kê MỨC BẢN GHI theo miền × lớp + Hình 1 / Record-level inventory by domain
# =========================================================
records = (files.groupby("record_uid")
           .agg(domain=("domain", "first"), record_id=("record_id", "first"),
                label=("label", "first"), cls=("cls", "first"),
                n_files=("file_stem", "size"), duration_s=("duration_s", "sum"),
                sr_native=("sr_native", "min"))
           .reset_index())
records.to_csv(OUT / "manifest_records.csv", index=False)
DOMAINS_ALL = sorted(records.domain.unique())
DOMAIN_COLOR = {d: DOMAIN_PALETTE[i % len(DOMAIN_PALETTE)] for i, d in enumerate(DOMAINS_ALL)}

inv = (records.groupby(["domain", "cls"])
       .agg(n_records=("record_uid", "nunique"), n_files=("n_files", "sum"),
            total_min=("duration_s", lambda s: s.sum() / 60), mean_s=("duration_s", "mean"),
            min_s=("duration_s", "min"), max_s=("duration_s", "max"))
       .reset_index())
save_table(inv.round(2), "inventory_domain_class",
           "Recording-level inventory per acquisition domain and class (a recording may span several files).", 2)

prior = records.groupby("domain").label.mean().rename("leak_fraction")
print("Tỉ lệ bản ghi leak theo miền:", prior.round(3).to_dict())
if prior.max() - prior.min() > 0.15:
    print("!! DỊCH CHUYỂN NHÃN: tỉ lệ lớp khác nhau > 15 điểm % giữa các miền. Căn chỉnh biên P(f)")
    print("   (DANN) có thể bị ảnh hưởng (Zhao et al., 2019) -> nguồn được lấy mẫu cân bằng lớp,")
    print("   và CDAN+E (căn chỉnh có điều kiện) là đối chứng trực tiếp cho vấn đề này.")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
piv = inv.pivot(index="domain", columns="cls", values="n_records").reindex(columns=["noleak", "leak"]).fillna(0)
xx = np.arange(len(piv)); w = .38
for j, c in enumerate(["noleak", "leak"]):
    ax[0].bar(xx + (j - .5) * w, piv[c].values, w, color=PAL[c], label=c)
    for x0, v in zip(xx + (j - .5) * w, piv[c].values):
        ax[0].text(x0, v, f"{int(v)}", ha="center", va="bottom", fontsize=8)
ax[0].set_xticks(xx); ax[0].set_xticklabels(piv.index, rotation=20)
ax[0].set_title("(a) Independent recordings"); ax[0].legend()
for d in DOMAINS_ALL:
    ax[1].hist(records.loc[records.domain == d, "duration_s"], bins=15, alpha=.55,
               color=DOMAIN_COLOR[d], label=d)
ax[1].set_xlabel("recording duration (s)"); ax[1].set_title("(b) Duration by domain"); ax[1].legend()
ax[2].bar(prior.index, prior.values, color=[DOMAIN_COLOR[d] for d in prior.index], width=.5)
ax[2].axhline(.5, color="k", lw=.8, ls="--")
ax[2].set_ylim(0, 1); ax[2].set_title("(c) Leak fraction (label shift)")
for i, v in enumerate(prior.values):
    ax[2].text(i, v, f"{v:.2f}", ha="center", va="bottom")
fig.suptitle("Mixed dataset: acquisition domains and class balance", y=1.04)
savefig(fig, "domain_inventory",
        "Dataset composition per acquisition domain: (a) independent recordings per class, "
        "(b) recording duration, (c) leak fraction, which quantifies label shift between domains.")

## 3. Phân tích dịch chuyển miền TRƯỚC khi huấn luyện / Domain-shift analysis before training

Mục này trả lời **RQ1** mà không dùng mạng sâu: (i) băng thông hiệu dụng mỗi miền, (ii) PSD theo miền × lớp,
(iii) dấu hiệu rò rỉ có **cùng chiều** giữa các miền không (điều kiện cần để UDA có ý nghĩa), (iv) trùng lặp
nội dung giữa các miền, (v) mức tách được của miền và **suy giảm chéo miền** của một bộ phân lớp cổ điển.

Toàn bộ tệp được đọc **một lần**: cùng lượt đó sinh frame (mục 4) và thống kê mức tệp (PSD Welch, băng thông,
đường bao năng lượng), lưu cache theo dấu vân tay nội dung.

In [ ]:
# =========================================================
# 3.1 Nạp + tiền xử lý + MỘT lượt quét: frame & thống kê tệp / single pass: frames + file statistics
# =========================================================
_HP, _NOTCH = {}, {}
def _hp_coeffs(sr, fc):
    k = (int(sr), float(fc))
    if k not in _HP:
        _HP[k] = sps.butter(4, float(fc) / (sr / 2), btype="high")
    return _HP[k]

def _notches(sr):
    k = int(sr)
    if k not in _NOTCH:
        _NOTCH[k] = [sps.iirnotch(cfg.mains_hz * h / (sr / 2), cfg.notch_q)
                     for h in range(1, cfg.mains_harmonics + 1) if cfg.mains_hz * h < sr / 2 - 10]
    return _NOTCH[k]

def load_audio(path, sr=None):
    sr = sr or cfg.target_sr
    try:
        x, _ = librosa.load(str(path), sr=sr, mono=True)
    except Exception:
        y, sr0 = sf.read(str(path), always_2d=True, dtype="float32")
        y = np.nan_to_num(y).mean(axis=1)
        x = librosa.resample(y, orig_sr=sr0, target_sr=sr) if sr0 != sr else y
    x = np.nan_to_num(np.asarray(x, dtype=np.float32))
    x = x - x.mean()
    if cfg.highpass_hz and cfg.highpass_hz > 0:
        b, a = _hp_coeffs(sr, cfg.highpass_hz); x = sps.filtfilt(b, a, x).astype(np.float32)
    if cfg.notch_mains:
        for b, a in _notches(sr):
            x = sps.filtfilt(b, a, x).astype(np.float32)
    x = np.nan_to_num(x)
    m = float(np.max(np.abs(x)))
    return (x / m if m > 1e-9 else x).astype(np.float32)

WELCH_NPERSEG = 4096
def file_stats(x, sr):
    nper = min(WELCH_NPERSEG, len(x))
    f, P = sps.welch(x, fs=sr, nperseg=nper, noverlap=nper // 2)
    if len(f) != WELCH_NPERSEG // 2 + 1:                     # tệp quá ngắn -> nội suy về lưới chuẩn
        fg = np.fft.rfftfreq(WELCH_NPERSEG, 1 / sr); P = np.interp(fg, f, P); f = fg
    tot = P.sum() + 1e-30
    cum = np.cumsum(P) / tot
    bw = float(f[min(len(f) - 1, np.searchsorted(cum, cfg.bw_energy_frac))])
    hop = int(0.1 * sr)
    env = np.log10(np.array([np.mean(x[i:i + hop] ** 2) + 1e-12 for i in range(0, len(x) - hop + 1, hop)]))
    return dict(psd=P.astype(np.float32), bw=bw, hf_ratio=float(P[f >= 4000].sum() / tot),
                env=env.astype(np.float32), rms=float(np.sqrt(np.mean(x ** 2))))

def _fingerprint(df):
    h = hashlib.sha1()
    for _, r in df.sort_values("path").iterrows():
        try:
            st = os.stat(r.path); meta = f"{st.st_size}|{int(st.st_mtime)}"
        except Exception:
            meta = "NA"
        h.update(f"{r.path}|{meta}|{r.record_uid}|{r.label}|".encode())
    h.update(json.dumps(dict(sr=cfg.target_sr, hp=cfg.highpass_hz, n=cfg.notch_mains, mh=cfg.mains_hz,
                             k=cfg.mains_harmonics, q=cfg.notch_q, fs=cfg.frame_sec, hs=cfg.hop_sec,
                             bwf=cfg.bw_energy_frac), sort_keys=True).encode())
    return h.hexdigest()[:12]

DATA_FP = _fingerprint(files)
_xp, _mp, _sp = (CACHE_DIR / f"frames_{DATA_FP}.npy", CACHE_DIR / f"frames_{DATA_FP}.csv",
                 CACHE_DIR / f"filestats_{DATA_FP}.pkl")
if _xp.exists() and _mp.exists() and _sp.exists():
    X_ALL = np.load(_xp, mmap_mode="r"); frames = pd.read_csv(_mp)
    FILE_STATS = pickle.load(open(_sp, "rb"))
    print("Nạp cache:", _xp.name)
else:
    t0 = time.time(); chunks, metas, FILE_STATS = [], [], {}
    for _, r in files.iterrows():
        x = load_audio(r.path)
        FILE_STATS[int(r.file_id)] = file_stats(x, cfg.target_sr)
        if len(x) < FRAME_LEN:
            x = np.pad(x, (0, FRAME_LEN - len(x)))
        for s in range(0, len(x) - FRAME_LEN + 1, HOP_LEN):
            seg = x[s:s + FRAME_LEN]
            chunks.append(seg)
            metas.append(dict(file_id=int(r.file_id), record_uid=r.record_uid, domain=r.domain,
                              label=int(r.label), cls=r.cls, start_s=s / cfg.target_sr,
                              rms=float(np.sqrt(np.mean(seg ** 2)))))
    X_ALL = np.stack(chunks).astype(np.float32); del chunks
    frames = pd.DataFrame(metas)
    np.save(_xp, X_ALL); frames.to_csv(_mp, index=False)
    pickle.dump(FILE_STATS, open(_sp, "wb"))
    X_ALL = np.load(_xp, mmap_mode="r")
    print(f"Quét {len(files)} tệp trong {time.time() - t0:.1f}s")

assert np.isfinite(np.asarray(X_ALL[:min(len(X_ALL), 2048)])).all(), "NaN/Inf trong frame"
y_all = frames.label.values.astype(np.float32)
uid_all = frames.record_uid.values
dom_all = frames.domain.values
N_FRAMES = len(frames)
PSD_F = np.fft.rfftfreq(WELCH_NPERSEG, 1 / cfg.target_sr)
print(f"Frame: {N_FRAMES} | shape {X_ALL.shape} | {X_ALL.nbytes / 1024 ** 3:.2f} GB")
display(frames.groupby(["domain", "cls"]).agg(n_frames=("label", "size"),
                                              n_records=("record_uid", "nunique"),
                                              mean_rms=("rms", "mean")).round(4))

In [ ]:
# =========================================================
# 3.2 Băng thông hiệu dụng theo miền + HOÀ HỢP BĂNG THÔNG / Effective bandwidth and harmonisation
# =========================================================
files["bw_hz"] = files.file_id.map(lambda i: FILE_STATS[i]["bw"])
files["hf_ratio"] = files.file_id.map(lambda i: FILE_STATS[i]["hf_ratio"])
rec_bw = files.groupby("record_uid").agg(domain=("domain", "first"), cls=("cls", "first"),
                                         bw_hz=("bw_hz", "median"), hf_ratio=("hf_ratio", "median"))
bw_tab = (rec_bw.groupby("domain")
          .agg(n_records=("bw_hz", "size"), bw_median_hz=("bw_hz", "median"),
               bw_p10_hz=("bw_hz", lambda s: s.quantile(.1)), bw_p90_hz=("bw_hz", lambda s: s.quantile(.9)),
               energy_above_4k=("hf_ratio", "median"))
          .reset_index())
save_table(bw_tab, "bandwidth_by_domain",
           f"Effective bandwidth per domain (frequency below which {cfg.bw_energy_frac*100:.1f}% of the "
           "signal energy lies; median over recordings) and median energy fraction above 4 kHz.", 3)

NYQ = cfg.target_sr / 2
F_MAX_EFF = float(min(cfg.f_max, NYQ))
HARMONISED = False
if cfg.bandwidth_harmonize == "auto":
    common = float(bw_tab.bw_p90_hz.min())
    if common < 0.85 * NYQ and bw_tab.bw_p90_hz.max() > 1.15 * common:
        F_MAX_EFF = float(max(1000.0, math.floor(common / 100) * 100)); HARMONISED = True
elif cfg.bandwidth_harmonize not in ("off", "", None):
    F_MAX_EFF = float(cfg.bandwidth_harmonize); HARMONISED = True
if HARMONISED:
    print(f"[P] HOÀ HỢP BĂNG THÔNG: mọi front-end phổ giới hạn ở 0-{F_MAX_EFF:.0f} Hz "
          f"(miền hẹp nhất có p90 băng thông {bw_tab.bw_p90_hz.min():.0f} Hz).")
    print("    Lý do: năng lượng trên dải này chỉ phản ánh CHUỖI THU của từng miền -> một dấu hiệu miền")
    print("    tầm thường mà bộ phân biệt miền sẽ khai thác. Viết rõ trong phần Methods.")
else:
    print(f"Băng thông các miền tương đương -> giữ f_max = {F_MAX_EFF:.0f} Hz.")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for i, d in enumerate(DOMAINS_ALL):
    v = rec_bw.loc[rec_bw.domain == d, "bw_hz"].values
    ax[0].scatter(np.full(len(v), i) + np.random.default_rng(i).uniform(-.15, .15, len(v)), v,
                  s=14, color=DOMAIN_COLOR[d], alpha=.7)
    ax[0].hlines(np.median(v), i - .3, i + .3, color="k", lw=2)
ax[0].axhline(F_MAX_EFF, color=PAL["warn"], ls="--", lw=1.2, label=f"front-end f_max = {F_MAX_EFF:.0f} Hz")
ax[0].set_xticks(range(len(DOMAINS_ALL))); ax[0].set_xticklabels(DOMAINS_ALL)
ax[0].set_ylabel("effective bandwidth (Hz)"); ax[0].set_title("(a) Effective bandwidth per recording")
ax[0].legend(loc="lower right")
for d in DOMAINS_ALL:
    v = rec_bw.loc[rec_bw.domain == d, "hf_ratio"].values
    ax[1].hist(np.log10(v + 1e-9), bins=20, alpha=.55, color=DOMAIN_COLOR[d], label=d)
ax[1].set_xlabel("log10(energy fraction above 4 kHz)"); ax[1].set_title("(b) High-frequency energy")
ax[1].legend()
savefig(fig, "bandwidth_by_domain",
        "Acquisition-chain fingerprint: (a) effective bandwidth of every recording by domain with the "
        "harmonised front-end limit; (b) distribution of the energy fraction above 4 kHz.")

In [ ]:
# =========================================================
# 3.3 PSD theo miền × lớp (mức bản ghi) / Record-level PSD per domain and class
# =========================================================
def record_psd_table():
    rows = {}
    for uid, g in files.groupby("record_uid"):
        P = np.mean([FILE_STATS[i]["psd"] for i in g.file_id], axis=0)
        rows[uid] = P / (P[(PSD_F >= cfg.f_min) & (PSD_F <= F_MAX_EFF)].sum() + 1e-30)   # chuẩn hoá: bất biến gain
    return rows
REC_PSD = record_psd_table()
_band = (PSD_F >= cfg.f_min) & (PSD_F <= F_MAX_EFF)

fig, axes = plt.subplots(1, len(DOMAINS_ALL) + 1, figsize=(4.2 * (len(DOMAINS_ALL) + 1), 3.6), sharey=True)
for ax, d in zip(axes, DOMAINS_ALL):
    for c in ["noleak", "leak"]:
        uids = records.loc[(records.domain == d) & (records.cls == c), "record_uid"]
        if not len(uids):
            continue
        L = 10 * np.log10(np.stack([REC_PSD[u] for u in uids])[:, _band] + 1e-20)
        mu = L.mean(0)
        se = L.std(0, ddof=1) / np.sqrt(len(L)) if len(L) > 1 else np.zeros(L.shape[1])
        ax.semilogx(PSD_F[_band], mu, color=PAL[c], lw=1.3, label=f"{c} (n={len(L)})")
        ax.fill_between(PSD_F[_band], mu - 1.96 * se, mu + 1.96 * se, color=PAL[c], alpha=.2)
    ax.set_title(f"domain: {d}"); ax.set_xlabel("frequency (Hz)"); ax.legend()
axes[0].set_ylabel("relative PSD (dB)")
ax = axes[-1]
for d in DOMAINS_ALL:
    for c, ls in [("noleak", "--"), ("leak", "-")]:
        uids = records.loc[(records.domain == d) & (records.cls == c), "record_uid"]
        if len(uids):
            L = 10 * np.log10(np.stack([REC_PSD[u] for u in uids])[:, _band] + 1e-20)
            ax.semilogx(PSD_F[_band], L.mean(0), color=DOMAIN_COLOR[d], ls=ls, lw=1.2, label=f"{d}/{c}")
ax.set_title("all domains overlaid"); ax.set_xlabel("frequency (Hz)"); ax.legend(fontsize=7)
savefig(fig, "psd_domain_class",
        "Record-level relative power spectral density (Welch) per domain and class (mean ± 95% CI over "
        "recordings). The right panel overlays domains: vertical offsets between domains of the same class "
        "visualise covariate shift.")

In [ ]:
# =========================================================
# 3.4-3.5 Năng lượng theo dải + AUC TRONG TỪNG MIỀN + kiểm tra CHIỀU dấu hiệu
#         Band energies, within-domain band AUC and cue-direction consistency (conditional-shift check)
# =========================================================
_edges = np.unique(np.round(np.geomspace(max(cfg.f_min, 20), F_MAX_EFF, 13)).astype(int))
BANDS = [(int(a), int(b)) for a, b in zip(_edges[:-1], _edges[1:])]

def band_energy(P):
    return np.array([P[(PSD_F >= a) & (PSD_F < b)].sum() for a, b in BANDS])

REC_BAND = pd.DataFrame({u: 10 * np.log10(band_energy(P) + 1e-20) for u, P in REC_PSD.items()}).T
REC_BAND.columns = [f"{a}-{b}" for a, b in BANDS]
REC_BAND = REC_BAND.join(records.set_index("record_uid")[["domain", "label"]])

def _auc_ci(y, s, n=500, seed=0):
    y = np.asarray(y); s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    a = roc_auc_score(y, s); rng = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            v.append(roc_auc_score(y[i], s[i]))
    return a, np.percentile(v, 2.5), np.percentile(v, 97.5)

band_rows = []
for d in DOMAINS_ALL:
    g = REC_BAND[REC_BAND.domain == d]
    for col in REC_BAND.columns[:-2]:
        a, lo, hi = _auc_ci(g.label.values, g[col].values)
        band_rows.append(dict(domain=d, band_hz=col, auc=a, lo=lo, hi=hi,
                              sig=bool(np.isfinite(lo) and (lo > .5 or hi < .5))))
band_df = pd.DataFrame(band_rows)
A = band_df.pivot(index="domain", columns="band_hz", values="auc")[REC_BAND.columns[:-2]]
S = band_df.pivot(index="domain", columns="band_hz", values="sig")[REC_BAND.columns[:-2]]

# Chiều dấu hiệu: một dải "nhất quán" nếu mọi miền cùng phía 0.5; "ĐỔI CHIỀU" nếu có hai miền
# có ý nghĩa (CI không chứa 0.5) ở hai phía khác nhau -> dịch chuyển có điều kiện thật sự.
cons = []
for col in A.columns:
    v = A[col].values - .5; sg = S[col].values
    flip = bool(((v > 0) & sg).any() and ((v < 0) & sg).any())
    same = bool(np.all(v > 0) or np.all(v < 0))
    cons.append(dict(band_hz=col, **{f"AUC_{d}": A.loc[d, col] for d in A.index},
                     direction=("FLIP (significant)" if flip else ("consistent" if same else "mixed"))))
cons_df = pd.DataFrame(cons)
save_table(cons_df, "band_auc_within_domain",
           "Univariate record-level AUC of each band's relative energy for leak vs no-leak, computed "
           "separately inside each domain. 'FLIP' marks bands whose cue reverses direction between domains "
           "with bootstrap CIs excluding 0.5 (evidence of conditional shift).", 3)
n_flip = int((cons_df.direction == "FLIP (significant)").sum())
n_cons = int((cons_df.direction == "consistent").sum())
print(f"Dải nhất quán chiều: {n_cons}/{len(cons_df)} | dải ĐỔI CHIỀU có ý nghĩa: {n_flip}")
if n_flip:
    print("!! Có dấu hiệu DỊCH CHUYỂN CÓ ĐIỀU KIỆN: cùng một dải báo hiệu rò rỉ theo hai chiều ngược nhau ở")
    print("   hai miền. Giả thiết P(y|x) chung của UDA bị vi phạm một phần -> phải thảo luận trong bài báo.")

fig, ax = plt.subplots(1, 2, figsize=(16, 1.2 + 0.6 * len(DOMAINS_ALL) + 1.6),
                       gridspec_kw=dict(width_ratios=[3, 2]))
im = ax[0].imshow(A.values - .5, cmap="RdBu_r", vmin=-.5, vmax=.5, aspect="auto")
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        ax[0].text(j, i, f"{A.values[i, j]:.2f}" + ("*" if S.values[i, j] else ""),
                   ha="center", va="center", fontsize=7)
ax[0].set_xticks(range(A.shape[1])); ax[0].set_xticklabels(A.columns, rotation=45, ha="right", fontsize=8)
ax[0].set_yticks(range(A.shape[0])); ax[0].set_yticklabels(A.index); ax[0].grid(False)
ax[0].set_title("(a) Within-domain band AUC (red: leak louder; * CI excludes 0.5)")
plt.colorbar(im, ax=ax[0], fraction=.025, pad=.01).set_label("AUC − 0.5", labelpad=2)
prof = REC_BAND.groupby(["domain", "label"])[list(A.columns)].mean()
prof = prof.sub(prof.mean(axis=1), axis=0)
for (d, lab), row in prof.iterrows():
    ax[1].plot(range(len(row)), row.values, color=DOMAIN_COLOR[d], ls="-" if lab == 1 else "--",
               marker="o", ms=3, label=f"{d}/{'leak' if lab == 1 else 'noleak'}")
ax[1].set_xticks(range(len(A.columns))); ax[1].set_xticklabels(A.columns, rotation=45, ha="right", fontsize=8)
ax[1].set_ylabel("band energy − mean (dB)"); ax[1].set_title("(b) Spectral-tilt profile"); ax[1].legend(fontsize=7)
savefig(fig, "band_auc_by_domain",
        "Is the leak cue shared across domains? (a) Record-level AUC of each band's relative energy inside each "
        "domain; (b) mean band-energy profiles (mean-removed) per domain and class.")

In [ ]:
# =========================================================
# 3.6 Phát hiện bản ghi TRÙNG NỘI DUNG / Content-duplicate detection (leakage guard)
#     So khớp đường bao log-năng lượng 100 ms (bền với khác biệt tần số lấy mẫu/lọc) giữa các
#     tệp thuộc HAI BẢN GHI KHÁC NHAU có thời lượng gần bằng nhau.
# =========================================================
DUP_PAIRS = pd.DataFrame()
if cfg.check_duplicates:
    def _z(v):
        return (v - v.mean()) / (v.std() + 1e-9)
    fl = files[["file_id", "record_uid", "domain", "duration_s", "rel"]].sort_values("duration_s").values
    dups = []
    for i in range(len(fl)):
        j = i + 1
        while j < len(fl) and fl[j][3] - fl[i][3] <= 1.0:
            if fl[i][1] != fl[j][1]:
                a, b = FILE_STATS[int(fl[i][0])]["env"], FILE_STATS[int(fl[j][0])]["env"]
                L = min(len(a), len(b))
                if L >= 20:
                    za, zb = _z(a[:L]), _z(b[:L])
                    r = max(float(np.dot(za[k:], zb[:L - k]) / (L - k)) for k in range(0, min(10, L - 10)))
                    r = max(r, max(float(np.dot(zb[k:], za[:L - k]) / (L - k)) for k in range(1, min(10, L - 10))))
                    if r >= 0.97:
                        dups.append(dict(file_a=fl[i][4], uid_a=fl[i][1], file_b=fl[j][4], uid_b=fl[j][1],
                                         cross_domain=fl[i][2] != fl[j][2], envelope_r=round(r, 4)))
            j += 1
    DUP_PAIRS = pd.DataFrame(dups)
    if len(DUP_PAIRS):
        save_table(DUP_PAIRS, "duplicate_pairs",
                   "Pairs of files from different recordings whose 100-ms energy envelopes correlate at r ≥ 0.97 "
                   "(likely the same acquisition stored twice).", 4)
        n_x = int(DUP_PAIRS.cross_domain.sum())
        print(f"!! {len(DUP_PAIRS)} cặp nghi trùng ({n_x} cặp GIỮA hai miền).")
        print("   Trùng giữa nguồn và đích = mô hình đã 'thấy' bản ghi đích có nhãn -> kết quả UDA bị thổi")
        print("   phồng. Hãy loại một bản trong mỗi cặp (cfg.exclude_domains hoặc xoá tệp) rồi chạy lại.")
    else:
        print("✓ Không phát hiện bản ghi trùng nội dung (r < 0.97 với mọi cặp ứng viên).")

In [ ]:
# =========================================================
# 3.7 Mức tách được của MIỀN và suy giảm CHÉO MIỀN với đặc trưng cổ điển
#     Domain separability (input-level PAD) and cross-domain degradation of a classical baseline
#     Đặc trưng: thống kê log-Mel mỗi frame (trung bình theo thời gian đã trừ mức trung bình + độ
#     lệch chuẩn theo thời gian) -> bất biến với gain, cùng dải f_max đã hoà hợp.
# =========================================================
_mel_cpu = torchaudio.transforms.MelSpectrogram(sample_rate=cfg.target_sr, n_fft=cfg.n_fft,
                                                hop_length=cfg.hop_length, n_mels=cfg.n_mels,
                                                f_min=cfg.f_min, f_max=F_MAX_EFF, power=2.0)
_mel_dev = copy.deepcopy(_mel_cpu).to(DEVICE)          # .to() của nn.Module là tại chỗ -> phải sao chép

def logmel_stats(idx, bs=512):
    out = []
    for s in range(0, len(idx), bs):
        xb = torch.from_numpy(np.asarray(X_ALL[np.asarray(idx[s:s + bs])])).to(DEVICE)
        with torch.no_grad():
            L = torch.log10(_mel_dev(xb) + 1e-10)
            mu = L.mean(-1); sd = L.std(-1)
            out.append(torch.cat([mu - mu.mean(1, keepdim=True), sd], 1).cpu().numpy())
    return np.concatenate(out)

FEAT_IN = logmel_stats(np.arange(N_FRAMES))
REC_FEAT = pd.DataFrame(FEAT_IN).groupby(uid_all).mean()
REC_FEAT = REC_FEAT.join(records.set_index("record_uid")[["domain", "label"]])
_fcols = [c for c in REC_FEAT.columns if c not in ("domain", "label")]

def lr_pipe():
    return LogisticRegression(C=0.1, max_iter=2000)

def cv_proba(Xm, y, k=5, seed=0, strat=None):
    # Xác suất out-of-fold mức bản ghi (phân tầng theo y hoặc theo strat)
    strat = y if strat is None else strat
    k = int(min(k, pd.Series(strat).value_counts().min()))
    p = np.full(len(y), np.nan)
    if k < 2:
        return p
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(Xm, strat):
        sc = StandardScaler().fit(Xm[tr])
        p[te] = lr_pipe().fit(sc.transform(Xm[tr]), y[tr]).predict_proba(sc.transform(Xm[te]))[:, 1]
    return p

def pad_from_err(err):
    return 2 * (1 - 2 * min(err, 0.5))

# (a) PAD đầu vào cho từng cặp miền (mức bản ghi)
pad_rows = []
for i, a in enumerate(DOMAINS_ALL):
    for b in DOMAINS_ALL[i + 1:]:
        g = REC_FEAT[REC_FEAT.domain.isin([a, b])]
        yd = (g.domain == b).astype(int).values
        p = cv_proba(g[_fcols].values, yd)
        err = float(np.mean((p >= .5) != yd))
        pad_rows.append(dict(domain_a=a, domain_b=b, n_a=int((yd == 0).sum()), n_b=int((yd == 1).sum()),
                             domain_clf_auc=roc_auc_score(yd, p), domain_clf_error=err, PAD=pad_from_err(err)))
PAD_INPUT = pd.DataFrame(pad_rows)
save_table(PAD_INPUT, "pad_input",
           "Input-level domain separability: a logistic-regression domain classifier on gain-invariant log-Mel "
           "statistics (record-level, stratified CV). PAD = 2(1 − 2ε); PAD → 2 means fully separable domains.", 3)

# (b) Ma trận huấn luyện miền A -> kiểm thử miền B (đường chéo = CV trong miền)
xd_rows = []
for a in DOMAINS_ALL:
    ga = REC_FEAT[REC_FEAT.domain == a]
    for b in DOMAINS_ALL:
        gb = REC_FEAT[REC_FEAT.domain == b]
        if ga.label.nunique() < 2 or gb.label.nunique() < 2:
            continue
        if a == b:
            p = cv_proba(ga[_fcols].values, ga.label.values)
            auc = roc_auc_score(ga.label.values, p) if np.isfinite(p).all() else np.nan
        else:
            sc = StandardScaler().fit(ga[_fcols].values)
            m = lr_pipe().fit(sc.transform(ga[_fcols].values), ga.label.values)
            auc = roc_auc_score(gb.label.values, m.predict_proba(sc.transform(gb[_fcols].values))[:, 1])
        xd_rows.append(dict(train=a, test=b, auc=auc))
XD = pd.DataFrame(xd_rows).pivot(index="train", columns="test", values="auc")
save_table(XD.reset_index().rename(columns={"train": "train \\ test"}), "cross_domain_matrix_classical",
           "Record-level AUC of a classical log-Mel-statistics + logistic-regression detector trained on one domain "
           "and tested on another (diagonal: within-domain stratified CV). Off-diagonal drops quantify the "
           "domain gap that motivates adaptation.", 3)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1, 1.3, 1.3]))
im = ax[0].imshow(XD.values, cmap="viridis", vmin=.5, vmax=1.0)
for i in range(XD.shape[0]):
    for j in range(XD.shape[1]):
        ax[0].text(j, i, f"{XD.values[i, j]:.2f}", ha="center", va="center",
                   color="w" if XD.values[i, j] < .8 else "k")
ax[0].set_xticks(range(XD.shape[1])); ax[0].set_xticklabels(XD.columns)
ax[0].set_yticks(range(XD.shape[0])); ax[0].set_yticklabels(XD.index)
ax[0].set_xlabel("test domain"); ax[0].set_ylabel("train domain"); ax[0].grid(False)
ax[0].set_title("(a) Classical detector AUC")
plt.colorbar(im, ax=ax[0], fraction=.046)
Z = PCA(2, random_state=0).fit_transform(StandardScaler().fit_transform(REC_FEAT[_fcols].values))
for d in DOMAINS_ALL:
    for lab, mk in [(0, "o"), (1, "^")]:
        m = (REC_FEAT.domain.values == d) & (REC_FEAT.label.values == lab)
        ax[1].scatter(Z[m, 0], Z[m, 1], s=22, marker=mk, color=DOMAIN_COLOR[d], alpha=.75,
                      label=f"{d}/{'leak' if lab else 'noleak'}")
ax[1].set_title("(b) PCA of record features (colour = domain)"); ax[1].legend(fontsize=7)
_sub = np.random.default_rng(0).choice(N_FRAMES, min(N_FRAMES, cfg.tsne_max_points), replace=False)
Zt = TSNE(2, init="pca", random_state=0, perplexity=min(30, max(5, len(_sub) // 4))).fit_transform(
    StandardScaler().fit_transform(FEAT_IN[_sub]))
for d in DOMAINS_ALL:
    for lab, mk in [(0, "o"), (1, "^")]:
        m = (dom_all[_sub] == d) & (y_all[_sub] == lab)
        ax[2].scatter(Zt[m, 0], Zt[m, 1], s=5, marker=mk, color=DOMAIN_COLOR[d], alpha=.5)
ax[2].set_title("(c) t-SNE of frame features (o noleak, ^ leak)"); ax[2].set_xticks([]); ax[2].set_yticks([])
savefig(fig, "input_domain_shift",
        "Input-level domain shift: (a) train-on-A/test-on-B AUC of a classical detector, (b) PCA of record-level "
        "log-Mel statistics and (c) t-SNE of frame-level statistics, coloured by domain.")

In [ ]:
# =========================================================
# 3.8 Spectrogram đại diện mỗi miền × lớp / Representative log-Mel spectrograms
# =========================================================
_rng = np.random.default_rng(1)
fig, axes = plt.subplots(2, len(DOMAINS_ALL), figsize=(4.4 * len(DOMAINS_ALL), 5.6), squeeze=False)
for j, d in enumerate(DOMAINS_ALL):
    for i, c in enumerate(["noleak", "leak"]):
        ax = axes[i, j]
        cand = frames.index[(frames.domain == d) & (frames.cls == c)].values
        if not len(cand):
            ax.axis("off"); continue
        k = int(_rng.choice(cand))
        with torch.no_grad():
            S = 10 * torch.log10(_mel_cpu(torch.from_numpy(np.asarray(X_ALL[k])).float()) + 1e-10).numpy()
        ax.imshow(S, origin="lower", aspect="auto", cmap="magma",
                  extent=[0, cfg.frame_sec, 0, cfg.n_mels], vmin=S.max() - 70, vmax=S.max())
        ax.set_title(f"{d} / {c}  ({frames.record_uid[k]})", fontsize=9); ax.grid(False)
        ax.set_xlabel("time (s)"); ax.set_ylabel("mel band")
savefig(fig, "spectrogram_examples",
        f"Representative log-Mel spectrograms (0–{F_MAX_EFF:.0f} Hz, {cfg.n_mels} bands) of one random frame "
        "per domain and class.")

## 4. Phân bố frame / Frame distribution

Frame `cfg.frame_sec` giây, bước `cfg.hop_sec` giây; **mọi frame mang `record_uid` và `domain`**. Số frame không làm tăng
cỡ mẫu hiệu dụng — đơn vị thống kê vẫn là bản ghi (v7, mục 4).

In [ ]:
# =========================================================
# 4.1 Hình: frame theo miền × lớp, mức RMS theo miền / frames per domain and RMS level
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
fc = frames.groupby(["domain", "cls"]).size().unstack().reindex(columns=["noleak", "leak"]).fillna(0)
fc.plot.bar(ax=ax[0], color=[PAL["noleak"], PAL["leak"]], rot=20, width=.7)
ax[0].set_title("(a) Frames per domain and class"); ax[0].set_xlabel("")
nfr = frames.groupby(["record_uid", "domain"]).size().reset_index(name="n")
for d in DOMAINS_ALL:
    ax[1].hist(nfr.loc[nfr.domain == d, "n"], bins=15, alpha=.55, color=DOMAIN_COLOR[d], label=d)
ax[1].set_title("(b) Frames per recording"); ax[1].legend()
for d in DOMAINS_ALL:
    for c, ls in [("noleak", "--"), ("leak", "-")]:
        v = 20 * np.log10(frames.loc[(frames.domain == d) & (frames.cls == c), "rms"] + 1e-9)
        if len(v) > 5:
            kde = spstats.gaussian_kde(v); xs = np.linspace(v.min(), v.max(), 200)
            ax[2].plot(xs, kde(xs), color=DOMAIN_COLOR[d], ls=ls, label=f"{d}/{c}")
ax[2].set_xlabel("frame RMS (dBFS, after per-file peak normalisation)"); ax[2].set_title("(c) Level")
ax[2].legend(fontsize=7)
savefig(fig, "frame_distribution",
        "Frame-level composition: (a) frames per domain and class, (b) frames per recording, "
        "(c) frame RMS level by domain and class. All networks normalise each input, so level is not usable.")

## 5. Kiến trúc mạng / Network architecture

| Khối | Cấu trúc | Đầu ra |
|---|---|---|
| Front-end (trong mô hình, chạy trên GPU) | log-Mel `n_mels`=64, `n_fft`=2048, hop 512 @ 16 kHz, `f_max` = băng thông chung (mục 3.2); SpecAugment **trước** chuẩn hoá; chuẩn hoá theo từng mẫu | 1×64×63 |
| $G_f$ — trích đặc trưng | 4 × [Conv3×3–BN–ReLU ×2, Squeeze-Excitation, MaxPool, Dropout2d 0.1], kênh 32/64/128/256 (giống `LeakCNN2D` v7) → GAP ⊕ GMP | 512 |
| Bottleneck | FC 512→256, BN, ReLU, Dropout 0.3 → **đặc trưng $f$ dùng để căn chỉnh miền** | 256 |
| $G_y$ — dự đoán nhãn | FC 256→1 (sigmoid) | xác suất rò rỉ |
| $G_d$ — phân biệt miền (DANN) | **GRL($-\lambda_p$)** → FC 256→256–ReLU–Dropout 0.5 → FC 256→256–ReLU–Dropout 0.5 → FC 256→1 | nguồn / đích |
| $G_d$ — CDAN(+E) | GRL → **$f \otimes g$** (256 × 2 = 512, $g=[1-p, p]$ tách gradient) → cùng MLP | nguồn / đích |
| Deep CORAL / MK-MMD | Không có $G_d$; thêm $\lambda_p \cdot$ CORAL($f_s, f_t$) hoặc MMD($f_s, f_t$) | — |

Dùng **cùng một $G_f$ + $G_y$** cho mọi phương pháp → khác biệt kết quả chỉ đến từ mục tiêu căn chỉnh miền.

In [ ]:
# =========================================================
# 5.1 Front-end log-Mel + SpecAugment (từ v7, f_max = băng thông chung)
# =========================================================
def _safe_norm(m):
    m = torch.nan_to_num(m, nan=0.0, posinf=0.0, neginf=-80.0)
    mu = m.mean(dim=(2, 3), keepdim=True)
    sd = m.std(dim=(2, 3), keepdim=True).clamp_min(1e-3)
    return torch.nan_to_num((m - mu) / sd, nan=0.0, posinf=0.0, neginf=0.0)

class SpecAugment(nn.Module):
    # Che dải tần / khoảng thời gian, MỖI MẪU một mặt nạ, giá trị che = mức im lặng của mẫu, chỉ khi train.
    def __init__(self, f=None, t=None, n=2):
        super().__init__()
        self.f = cfg.specaug_freq_mask if f is None else int(f)
        self.t = cfg.specaug_time_mask if t is None else int(t)
        self.n = int(n)
        self.generator = None
    def _band(self, B, L, W, device):
        g = self.generator
        w = torch.randint(0, W + 1, (B,), generator=g)
        s0 = (torch.rand(B, generator=g) * (L - w).clamp_min(1).float()).long()
        ar = torch.arange(L)
        return ((ar[None] >= s0[:, None]) & (ar[None] < (s0 + w)[:, None])).to(device)
    def forward(self, s):
        if not self.training or (self.f <= 0 and self.t <= 0):
            return s
        B, C, Fq, Tm = s.shape
        fill = s.amin(dim=(2, 3), keepdim=True).detach()
        mask = torch.zeros(B, 1, Fq, Tm, dtype=torch.bool, device=s.device)
        for _ in range(self.n):
            if self.f > 0 and Fq > self.f:
                mask = mask | self._band(B, Fq, self.f, s.device)[:, None, :, None]
            if self.t > 0 and Tm > self.t:
                mask = mask | self._band(B, Tm, self.t, s.device)[:, None, None, :]
        return torch.where(mask, fill, s)

class LogMelFrontend(nn.Module):
    def __init__(self, specaug=None):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.target_sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length, n_mels=cfg.n_mels,
            f_min=cfg.f_min, f_max=F_MAX_EFF, power=2.0, center=True)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)
        self.specaug = specaug
    def forward(self, x):
        with amp_autocast(x.device, False):                 # front-end luôn fp32 (tránh tràn fp16)
            s = self.to_db(self.mel(x.float())).unsqueeze(1)
        if self.specaug is not None:
            s = self.specaug(s)
        return _safe_norm(s)

with torch.no_grad():
    _s = LogMelFrontend()(torch.from_numpy(np.asarray(X_ALL[:2])).float())
SPEC_SHAPE = tuple(_s.shape[1:])
print("log-Mel:", SPEC_SHAPE, f"| 0-{F_MAX_EFF:.0f} Hz")

In [ ]:
# =========================================================
# 5.2 G_f + bottleneck + G_y, GRL, bộ phân biệt miền, CDAN, CORAL, MK-MMD
# =========================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, max(4, ch // r)), nn.ReLU(inplace=True),
                                nn.Linear(max(4, ch // r), ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=(2, 3)))[:, :, None, None]

class ConvBlock2D(nn.Module):
    def __init__(self, cin, cout, pool=(2, 2), p_drop=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.se = SEBlock(cout)
        self.pool = nn.MaxPool2d(pool)
        self.drop = nn.Dropout2d(p_drop)
    def forward(self, x):
        return self.drop(self.pool(self.se(self.body(x))))

def gap_gmp(h):
    dims = tuple(range(2, h.dim()))
    return torch.cat([h.mean(dim=dims), h.amax(dim=dims)], dim=1)

class LeakNet(nn.Module):
    """G_f (front-end + CNN) -> bottleneck f -> G_y. Dùng chung cho MỌI phương pháp."""
    def __init__(self, backbone=None):
        super().__init__()
        self.backbone = backbone or cfg.backbone
        if self.backbone == "cnn2d":
            self.specaug = SpecAugment()
            self.front = LogMelFrontend(specaug=self.specaug)
            blocks, cin = [], 1
            for i, w in enumerate(cfg.widths):
                blocks.append(ConvBlock2D(cin, w, pool=(2, 2) if i < len(cfg.widths) - 1 else (2, 1)))
                cin = w
            self.blocks = nn.Sequential(*blocks)
            d_trunk = 2 * cin
        else:
            c1, c2, c3, c4 = [2 * w for w in cfg.widths]
            self.net1d = nn.Sequential(
                nn.Conv1d(1, c1, 80, stride=4, padding=38, bias=False), nn.BatchNorm1d(c1), nn.ReLU(True),
                nn.MaxPool1d(4),
                nn.Conv1d(c1, c2, 3, padding=1, bias=False), nn.BatchNorm1d(c2), nn.ReLU(True), nn.MaxPool1d(4),
                nn.Conv1d(c2, c3, 3, padding=1, bias=False), nn.BatchNorm1d(c3), nn.ReLU(True), nn.MaxPool1d(4),
                nn.Conv1d(c3, c4, 3, padding=1, bias=False), nn.BatchNorm1d(c4), nn.ReLU(True))
            d_trunk = 2 * c4
        self.bottleneck = nn.Sequential(nn.Linear(d_trunk, cfg.bottleneck_dim),
                                        nn.BatchNorm1d(cfg.bottleneck_dim), nn.ReLU(inplace=True),
                                        nn.Dropout(0.3))
        self.cls = nn.Linear(cfg.bottleneck_dim, 1)

    def spec(self, x):
        return self.front(x)
    def trunk_from_spec(self, s):
        return gap_gmp(self.blocks(s))
    def trunk(self, x):
        if self.backbone == "cnn2d":
            return self.trunk_from_spec(self.front(x))
        x = x.float()
        if HARMONISED:                                         # cùng giới hạn băng thông với front-end phổ
            with amp_autocast(x.device, False):
                x = torchaudio.functional.lowpass_biquad(x, cfg.target_sr, F_MAX_EFF)
        x = (x - x.mean(-1, keepdim=True)) / x.std(-1, keepdim=True).clamp_min(1e-4)
        return gap_gmp(self.net1d(x.unsqueeze(1)))
    def head(self, z):
        f = self.bottleneck(z)
        return self.cls(f).squeeze(-1), f
    def forward(self, x, return_feat=False):
        logit, f = self.head(self.trunk(x))
        return (logit, f) if return_feat else logit


class GradReverse(torch.autograd.Function):
    # Lớp đảo gradient (Ganin & Lempitsky, 2015): xuôi = đồng nhất, ngược = -coeff * gradient.
    @staticmethod
    def forward(ctx, x, coeff):
        ctx.coeff = float(coeff)
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.coeff * g, None

class DomainDisc(nn.Module):
    def __init__(self, din, hidden=None, p=None):
        super().__init__()
        h = hidden or cfg.disc_hidden; p = cfg.disc_dropout if p is None else p
        self.net = nn.Sequential(nn.Linear(din, h), nn.ReLU(inplace=True), nn.Dropout(p),
                                 nn.Linear(h, h), nn.ReLU(inplace=True), nn.Dropout(p), nn.Linear(h, 1))
    def forward(self, h, coeff=1.0):
        return self.net(GradReverse.apply(h, coeff)).squeeze(-1)

def cdan_input(f, logit):
    # Ánh xạ đa tuyến f ⊗ g (Long et al., 2018); g = [1-p, p] được TÁCH gradient như bản gốc.
    p = torch.sigmoid(logit.float()).detach()
    g = torch.stack([1 - p, p], dim=1).to(f.dtype)
    return torch.bmm(g.unsqueeze(2), f.unsqueeze(1)).flatten(1)

def binary_entropy(logit):
    p = torch.sigmoid(logit.float()).clamp(1e-6, 1 - 1e-6)
    return -(p * torch.log(p) + (1 - p) * torch.log(1 - p))

def entropy_weights(logit_s, logit_t):
    # CDAN+E: w = 1 + exp(-H), chuẩn hoá RIÊNG trong nửa nguồn và nửa đích (mỗi miền tổng trọng số = 1).
    ws = 1 + torch.exp(-binary_entropy(logit_s)); wt = 1 + torch.exp(-binary_entropy(logit_t))
    return torch.cat([ws / ws.sum(), wt / wt.sum()]).detach()

def coral_loss(fs, ft):
    # Deep CORAL (Sun & Saenko, 2016): ||C_s - C_t||_F^2 / (4 d^2)
    fs, ft = fs.float(), ft.float(); d = fs.size(1)
    def cov(x):
        x = x - x.mean(0, keepdim=True)
        return x.t() @ x / max(x.size(0) - 1, 1)
    return ((cov(fs) - cov(ft)) ** 2).sum() / (4 * d * d)

def mmd_loss(fs, ft, mults=(0.25, 0.5, 1.0, 2.0, 4.0)):
    # MK-MMD (Long et al., 2015), ước lượng chệch, băng thông theo trung bình khoảng cách.
    x = torch.cat([fs, ft]).float(); n = x.size(0); ns = fs.size(0)
    d2 = torch.cdist(x, x).pow(2)
    bw = d2.detach().sum() / max(n * n - n, 1)
    K = sum(torch.exp(-d2 / (bw * m + 1e-9)) for m in mults)
    return K[:ns, :ns].mean() + K[ns:, ns:].mean() - 2 * K[:ns, ns:].mean()

# ---- Kiểm thử đơn vị / unit tests ----------------------------------------------------
_x = torch.randn(4, 3, requires_grad=True)
GradReverse.apply(_x, 0.5).sum().backward()
assert torch.allclose(_x.grad, torch.full_like(_x, -0.5)), "GRL phải nhân gradient với -coeff"
_a = torch.randn(64, 16)
assert float(coral_loss(_a, _a)) < 1e-10 and float(mmd_loss(_a, _a)) < 1e-6
assert float(coral_loss(_a, 3 * _a + 1)) > 0 and float(mmd_loss(_a, _a + 2)) > 0
_net = LeakNet(); _lg, _f = _net(torch.from_numpy(np.asarray(X_ALL[:3])).float(), return_feat=True)
assert _lg.shape == (3,) and _f.shape == (3, cfg.bottleneck_dim)
assert cdan_input(_f, _lg).shape == (3, 2 * cfg.bottleneck_dim)
_w = entropy_weights(_lg[:2], _lg[2:])
assert abs(float(_w.sum()) - 2.0) < 1e-5
print("✓ GRL, CORAL, MMD, CDAN, trọng số entropy: kiểm thử đơn vị đạt.")

In [ ]:
# =========================================================
# 5.3 Bảng tham số + Hình: sơ đồ kiến trúc / Parameter table and architecture diagram
# =========================================================
def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))

_net = LeakNet()
_trunk = n_params(_net.blocks) if cfg.backbone == "cnn2d" else n_params(_net.net1d)
arch = pd.DataFrame([
    dict(component="G_f feature extractor (CNN)", params=_trunk, output=f"{2 * cfg.widths[-1]}-d (GAP⊕GMP)"),
    dict(component="Bottleneck (FC-BN-ReLU-Dropout)", params=n_params(_net.bottleneck), output=f"{cfg.bottleneck_dim}-d f"),
    dict(component="G_y label predictor", params=n_params(_net.cls), output="1 logit"),
    dict(component="G_d domain discriminator (DANN)", params=n_params(DomainDisc(cfg.bottleneck_dim)),
         output="1 logit (source/target)"),
    dict(component="G_d domain discriminator (CDAN, f⊗g)", params=n_params(DomainDisc(2 * cfg.bottleneck_dim)),
         output="1 logit (source/target)"),
])
arch["params_M"] = (arch.params / 1e6).round(3)
save_table(arch, "architecture_parameters",
           f"Network components and trainable parameters (backbone '{cfg.backbone}', input {cfg.frame_sec:g}-s frame "
           f"at {cfg.target_sr} Hz, log-Mel {SPEC_SHAPE[1]}x{SPEC_SHAPE[2]}).", 3)
del _net

def _box(ax, xy, w, h, text, fc, ls="-"):
    ax.add_patch(FancyBboxPatch(xy, w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                fc=fc, ec="#333333", lw=1.1, ls=ls))
    ax.text(xy[0] + w / 2, xy[1] + h / 2, text, ha="center", va="center", fontsize=8.2)

def _arrow(ax, a, b, color="#333333", ls="-"):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2, ls=ls))

fig, ax = plt.subplots(figsize=(15.5, 4.6))
ax.set_xlim(0, 15.3); ax.set_ylim(0, 5); ax.axis("off")
_box(ax, (0.1, 2.0), 1.6, 1.0, f"Waveform\n{cfg.frame_sec:g} s @ {cfg.target_sr // 1000} kHz\nsource + target", "#EEEEEE")
_box(ax, (2.1, 2.0), 2.5, 1.0, f"log-Mel front-end\n{SPEC_SHAPE[1]}×{SPEC_SHAPE[2]}, 0–{F_MAX_EFF / 1000:.1f} kHz\nSpecAugment + per-sample norm", "#DCE9F2")
_box(ax, (5.0, 2.0), 2.6, 1.0, "Feature extractor $G_f$\n4×[Conv-BN-ReLU×2, SE, Pool]\nGAP ⊕ GMP", "#DCE9F2")
_box(ax, (8.0, 2.0), 1.8, 1.0, f"Bottleneck\nFC-BN-ReLU-Drop\nf ∈ R^{cfg.bottleneck_dim}", "#DCE9F2")
_box(ax, (10.5, 3.6), 2.0, 0.9, "Label predictor $G_y$\nFC → sigmoid", "#F6E0D9")
_box(ax, (13.0, 3.6), 2.1, 0.9, "$\\mathcal{L}_y$: BCE\n(source labels)", "#FFFFFF")
_box(ax, (10.2, 0.4), 1.0, 0.9, "GRL\n$-\\lambda_p$", "#F3E3C3")
_box(ax, (11.5, 0.4), 1.5, 0.9, "CDAN: f ⊗ g\n(optional)", "#FFFFFF", ls="--")
_box(ax, (13.3, 0.4), 1.9, 0.9, "Domain disc. $G_d$\nMLP 256-256-1", "#E4DDF0")
_arrow(ax, (1.7, 2.5), (2.1, 2.5)); _arrow(ax, (4.6, 2.5), (5.0, 2.5)); _arrow(ax, (7.6, 2.5), (8.0, 2.5))
_arrow(ax, (9.8, 2.7), (10.5, 4.0)); _arrow(ax, (12.5, 4.05), (13.0, 4.05))
_arrow(ax, (9.8, 2.3), (10.2, 0.9)); _arrow(ax, (11.2, 0.85), (11.5, 0.85)); _arrow(ax, (13.0, 0.85), (13.3, 0.85))
_arrow(ax, (11.5, 3.6), (12.25, 1.3), color="#999999", ls="--")
ax.text(14.25, 0.12, "$\\mathcal{L}_d$: source vs target", ha="center", fontsize=8)
ax.text(8.9, 3.3, "Deep CORAL / MK-MMD act on f directly", ha="center", fontsize=7.5, color="#555555")
ax.text(9.75, 1.15, "reversed gradient\n$-\\lambda_p\\,\\partial\\mathcal{L}_d/\\partial f$", ha="right",
        fontsize=7.5, color="#7a5a10")
savefig(fig, "architecture_dann",
        "Adversarial domain adaptation network: shared feature extractor G_f and bottleneck f, label predictor G_y "
        "trained on labelled source frames, and domain discriminator G_d connected through a gradient reversal layer "
        "(GRL). CDAN conditions G_d on the multilinear map f ⊗ g of features and class probabilities.")

## 6. Bài toán chuyển miền và giao thức đánh giá / Transfer tasks and evaluation protocol

Với mỗi bài toán `nguồn → đích` và mỗi seed:

| Tập | Gồm | Dùng để |
|---|---|---|
| `src_train` | ~80% bản ghi nguồn (phân tầng theo miền × lớp) | học $G_y$ (có nhãn) |
| `src_val` | ~20% bản ghi nguồn | chọn epoch (quy tắc `src_val`), ngưỡng nguồn, ước lượng DEV |
| `tgt_train` (fold *k*) | bản ghi đích **ngoài** fold *k* — **không nhãn** | căn chỉnh miền (DANN, CDAN+E, CORAL, MMD, AdaBN) |
| `tgt_test` (fold *k*) | bản ghi đích **trong** fold *k* | **chỉ để báo cáo** |

- **Quy nạp**: mỗi bản ghi đích nằm trong `tgt_test` **đúng một lần** và **không bao giờ** có mặt (kể cả không nhãn)
  trong lượt huấn luyện sinh ra dự đoán của nó. `source_only` không dùng đích nên được huấn luyện **một lần mỗi seed**
  và dự đoán toàn bộ bản ghi đích (tương đương OOF, rẻ hơn K lần).
- `oracle` dùng **nhãn** của `tgt_train` (cùng fold) → cận trên tham chiếu, không phải UDA.
- **5 assertion cứng** được kiểm tra cho mọi phân hoạch: không giao nhau giữa 4 tập, phủ đích đúng một lần, đủ hai lớp.

In [ ]:
# =========================================================
# 6.1 Xác định bài toán chuyển miền / Define transfer tasks
# =========================================================
rec_counts = records.groupby(["domain", "label"]).size().unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
HAS_BOTH = [d for d in DOMAINS_ALL if rec_counts.loc[d].min() >= 1]
ELIGIBLE_TGT = [d for d in DOMAINS_ALL if rec_counts.loc[d].min() >= cfg.min_records_per_class]

def _parse_task(spec):
    src, tgt = spec
    return [s.strip() for s in str(src).split("+") if s.strip()], str(tgt).strip()

if cfg.tasks:
    _raw = [_parse_task(t) for t in cfg.tasks]
else:
    _raw = [([d for d in HAS_BOTH if d != t], t) for t in ELIGIBLE_TGT]
TASKS = []
for srcs, tgt in _raw:
    bad = [d for d in srcs + [tgt] if d not in DOMAINS_ALL]
    assert not bad, f"Miền không tồn tại: {bad}. Các miền: {DOMAINS_ALL}"
    assert tgt not in srcs, f"Miền đích {tgt} không được nằm trong nguồn"
    if not srcs:
        continue
    src_lab = records[records.domain.isin(srcs)].label
    if src_lab.nunique() < 2 or rec_counts.loc[tgt].min() < cfg.min_records_per_class:
        print(f"  bỏ bài toán {'+'.join(srcs)} → {tgt}: thiếu bản ghi một lớp"); continue
    TASKS.append(dict(i=len(TASKS), src=srcs, tgt=tgt, name=f"{'+'.join(srcs)} → {tgt}",
                      slug=re.sub(r"[^A-Za-z0-9]+", "-", f"{'+'.join(srcs)}-to-{tgt}").strip("-")))
TASKS = TASKS[:cfg.max_tasks]
assert TASKS, "Không có bài toán chuyển miền hợp lệ (mỗi miền đích cần đủ bản ghi hai lớp)."

_tt = []
for T in TASKS:
    s = records[records.domain.isin(T["src"])]; t = records[records.domain == T["tgt"]]
    _tt.append(dict(task=T["name"], src_records=len(s), src_leak=int(s.label.sum()), tgt_records=len(t),
                    tgt_leak=int(t.label.sum()), src_frames=int(np.isin(dom_all, T["src"]).sum()),
                    tgt_frames=int((dom_all == T["tgt"]).sum()),
                    tgt_folds=int(min(cfg.tgt_folds, t.label.value_counts().min()))))
TASK_TAB = pd.DataFrame(_tt)
save_table(TASK_TAB, "transfer_tasks",
           "Transfer tasks (source → target), number of independent recordings (leak count) and frames per side, "
           "and number of recording-level folds on the target domain.", 0)

In [ ]:
# =========================================================
# 6.2 Phân hoạch quy nạp + 5 assertion cứng / Inductive partitions with hard checks
# =========================================================
FR_BY_UID = {k: np.asarray(v, dtype=np.int64) for k, v in frames.groupby("record_uid").indices.items()}
REC_LAB = records.set_index("record_uid").label

def frames_of(uids):
    uids = list(uids)
    return np.sort(np.concatenate([FR_BY_UID[u] for u in uids])) if uids else np.array([], dtype=np.int64)

def build_partition(T, seed):
    src = records[records.domain.isin(T["src"])].reset_index(drop=True)
    tgt = records[records.domain == T["tgt"]].reset_index(drop=True)
    n_val = max(2, int(round(cfg.src_val_frac * len(src))))
    strat = (src.domain + "|" + src.label.astype(str)).values
    if pd.Series(strat).value_counts().min() < 2 or n_val < len(set(strat)):
        strat = src.label.values                       # quá ít bản ghi để phân tầng theo miền × lớp
    a, b = next(StratifiedShuffleSplit(1, test_size=n_val, random_state=int(seed)).split(np.zeros(len(src)), strat))
    s_tr, s_va = src.record_uid.values[a], src.record_uid.values[b]
    t_uids = tgt.record_uid.values
    K = int(min(cfg.tgt_folds, tgt.label.value_counts().min()))
    if cfg.target_protocol == "transductive" or K < 2:
        mode, folds = "transductive", [(t_uids, t_uids)]
    else:
        mode = "inductive"
        skf = StratifiedKFold(K, shuffle=True, random_state=int(seed))
        folds = [(t_uids[i], t_uids[j]) for i, j in skf.split(np.zeros(len(tgt)), tgt.label.values)]
    P = dict(task=T["i"], seed=int(seed), mode=mode, src_tr=s_tr, src_va=s_va, tgt_all=t_uids, folds=folds,
             src_tr_idx=frames_of(s_tr), src_va_idx=frames_of(s_va), tgt_all_idx=frames_of(t_uids),
             fold_idx=[(frames_of(i), frames_of(j)) for i, j in folds])
    check_partition(P)
    return P

def check_partition(P):
    s_tr, s_va, t_all = set(P["src_tr"]), set(P["src_va"]), set(P["tgt_all"])
    assert not (s_tr & s_va), "LEAKAGE: bản ghi nằm ở cả src_train và src_val"
    assert not ((s_tr | s_va) & t_all), "LEAKAGE: bản ghi nằm ở cả nguồn và đích"
    assert REC_LAB[list(s_tr)].nunique() == 2 and REC_LAB[list(s_va)].nunique() == 2, "nguồn thiếu một lớp"
    seen = []
    for k, (tr, te) in enumerate(P["folds"]):
        if P["mode"] == "inductive":
            assert not (set(tr) & set(te)), f"LEAKAGE: fold {k} có bản ghi đích ở cả tgt_train và tgt_test"
        assert REC_LAB[list(te)].nunique() == 2, f"fold {k}: tgt_test thiếu một lớp"
        seen += list(te)
    if P["mode"] == "inductive":
        assert sorted(seen) == sorted(t_all), "mỗi bản ghi đích phải ở tgt_test ĐÚNG MỘT LẦN"
    return True

PARTS = {(T["i"], s): build_partition(T, s) for T in TASKS for s in cfg.seeds}
print(f"✓ {len(PARTS)} phân hoạch (bài toán × seed) qua 5 assertion; chế độ: "
      f"{sorted({P['mode'] for P in PARTS.values()})}")

_pr = []
for T in TASKS:
    P = PARTS[(T["i"], cfg.seeds[0])]
    for k, (tr, te) in enumerate(P["folds"]):
        _pr.append(dict(task=T["name"], fold=k, src_train_rec=len(P["src_tr"]), src_val_rec=len(P["src_va"]),
                        tgt_unlab_rec=len(tr), tgt_test_rec=len(te),
                        tgt_test_leak=int(REC_LAB[list(te)].sum()),
                        tgt_unlab_frames=len(P["fold_idx"][k][0]), tgt_test_frames=len(P["fold_idx"][k][1])))
save_table(pd.DataFrame(_pr), "partitions",
           f"Recording-level partitions for seed {cfg.seeds[0]} (inductive protocol: the unlabelled target data used "
           "for adaptation never contains the recordings being tested).", 0)

# ---- Hình: bản đồ phân hoạch của bài toán đầu tiên ----
T0, P0 = TASKS[0], PARTS[(0, cfg.seeds[0])]
cols = list(P0["src_tr"]) + list(P0["src_va"]) + list(P0["tgt_all"])
order = sorted(range(len(cols)), key=lambda i: (cols[i] not in set(P0["src_tr"]) | set(P0["src_va"]),
                                                REC_LAB[cols[i]], cols[i]))
cols = [cols[i] for i in order]; pos = {u: i for i, u in enumerate(cols)}
M = np.full((len(P0["folds"]), len(cols)), np.nan)
for k, (tr, te) in enumerate(P0["folds"]):
    for u in P0["src_tr"]: M[k, pos[u]] = 0
    for u in P0["src_va"]: M[k, pos[u]] = 1
    for u in tr: M[k, pos[u]] = 2
    for u in te: M[k, pos[u]] = 3
cmap = matplotlib.colors.ListedColormap(["#9FB7C9", "#2E6F95", "#F1C9B9", "#C8553D"])
fig, ax = plt.subplots(2, 1, figsize=(14, 1.2 + 0.45 * len(P0["folds"])), sharex=True,
                       gridspec_kw=dict(height_ratios=[max(2, len(P0["folds"])), .7], hspace=.12))
ax[0].imshow(M, aspect="auto", cmap=cmap, vmin=-.5, vmax=3.5, interpolation="nearest")
ax[0].set_yticks(range(len(P0["folds"]))); ax[0].set_yticklabels([f"fold {k}" for k in range(len(P0["folds"]))])
ax[0].grid(False)
ax[0].set_title(f"{T0['name']}: source-train | source-val | target unlabelled (adaptation) | target test")
ax[1].imshow(np.array([[REC_LAB[u] for u in cols]]), aspect="auto",
             cmap=matplotlib.colors.ListedColormap([PAL["noleak"], PAL["leak"]]), interpolation="nearest")
ax[1].set_yticks([0]); ax[1].set_yticklabels(["label"]); ax[1].set_xlabel("recording"); ax[1].grid(False)
savefig(fig, "partition_map",
        "Inductive evaluation protocol for the first transfer task: every target recording is tested exactly once, "
        "by a model whose unlabelled adaptation data excluded it (colours: light/dark blue = source train/validation, "
        "light/dark red = target unlabelled/test).")

## 7. Bộ máy huấn luyện / Training engine

- **Một vòng lặp** = một lô nguồn (cân bằng lớp) + một lô đích không nhãn (+ một lô đích có nhãn với `oracle`/bán giám sát);
  hai miền đi **chung một lượt truyền** để BatchNorm thấy cả hai phân bố (chuẩn của DANN/CDAN).
- **Lịch** (Ganin et al., 2016): $\lambda_p = \lambda_{\max}\left(\frac{2}{1+e^{-10p}} - 1\right)$; LR: warm-up 5% + cosine (như v7);
  AdamW, AMP fp16, cắt gradient 5.0; nhãn làm mịn 0.05.
- **Đánh giá mỗi `eval_every` epoch** trên: `src_val` (BCE, AUC, ngưỡng Youden nguồn), `tgt_test` (lưu xác suất),
  và một mẫu con của `src_train`/`tgt_train` để tính **entropy đích**, **DEV** (You et al., 2019) và **PAD** của đặc trưng.
- Nhãn đích chỉ dùng cho cột *giám sát* (`tgt_auc_*`) và quy tắc `oracle` — **không bao giờ** cho quy tắc chính.
- Hai GPU: mỗi GPU chạy **một job độc lập** (không `DataParallel` → thống kê BatchNorm không bị chia đôi).

In [ ]:
# =========================================================
# 7.1 Dữ liệu thường trú GPU, bộ lấy mẫu, augmentation / resident data, samplers, augmentation
# =========================================================
DATA_GB = X_ALL.nbytes / 1024 ** 3
_RES, _RES_LOCK, _BUILD_LOCK = {}, threading.Lock(), threading.Lock()

def get_resident(device):
    if device.type != "cuda":
        return None
    key = str(device)
    with _RES_LOCK:
        if key not in _RES:
            if DATA_GB <= cfg.gpu_resident_max_gb:
                dt = torch.float32
            elif DATA_GB / 2 <= cfg.gpu_resident_max_gb:
                dt = torch.float16
            else:
                _RES[key] = None
                print(f"  [{device}] dữ liệu {DATA_GB:.1f} GB > ngưỡng -> đọc từ RAM mỗi lô")
                return None
            R = torch.empty(X_ALL.shape, dtype=dt, device=device)
            for s in range(0, len(X_ALL), 4096):
                R[s:s + 4096] = torch.from_numpy(np.asarray(X_ALL[s:s + 4096])).to(device, dtype=dt)
            _RES[key] = R
            print(f"  [{device}] nạp {DATA_GB * (1 if dt == torch.float32 else .5):.2f} GB frame ({dt})")
    return _RES[key]

def wave_augment(xb, g, hp):
    # Đáp ứng kênh FIR ngẫu nhiên | nhiễu Gauss theo SNR | dịch vòng — theo lô, trên thiết bị của lô.
    B, T = xb.shape; dev = xb.device
    rnd = lambda n: torch.rand(n, device=dev, generator=g)
    m = rnd(B) < hp["fir_p"]
    if bool(m.any()):
        taps = cfg.aug_fir_taps + (1 - cfg.aug_fir_taps % 2)
        h = torch.randn(B, 1, taps, device=dev, generator=g) * torch.hann_window(taps, periodic=False, device=dev)
        h = h / h.norm(dim=2, keepdim=True).clamp_min(1e-12)
        delta = torch.zeros(B, 1, taps, device=dev); delta[:, 0, taps // 2] = 1.0
        h = torch.where(m.view(B, 1, 1), h, delta)
        xb = F.conv1d(xb.view(1, B, T), h, padding=taps // 2, groups=B).view(B, T)
    m = (rnd(B) < hp["noise_p"]).float()
    lo, hi = cfg.aug_noise_snr_db
    snr = rnd(B) * (hi - lo) + lo
    sigma = torch.sqrt(((xb ** 2).mean(1) + 1e-12) / (10 ** (snr / 10.0))) * m
    xb = xb + torch.randn(xb.shape, device=dev, generator=g) * sigma[:, None]
    m = rnd(B) < hp["shift_p"]
    if bool(m.any()):
        sh = torch.randint(0, T, (B,), device=dev, generator=g) * m
        xb = torch.gather(xb, 1, (torch.arange(T, device=dev)[None] - sh[:, None]) % T)
    return torch.nan_to_num(xb)

def fetch(idx, device, augment=False, g=None, hp=None):
    idx = np.asarray(idx, dtype=np.int64)
    R = get_resident(device)
    if R is not None:
        xb = R.index_select(0, torch.as_tensor(idx, device=device)).float()
    else:
        xb = torch.from_numpy(np.asarray(X_ALL[idx], dtype=np.float32)).to(device, non_blocking=True)
    return wave_augment(xb, g, hp) if augment else xb

class Sampler:
    # Lấy mẫu "vô hạn" theo hoán vị (phủ đều dữ liệu); balanced=True -> nửa leak nửa noleak.
    def __init__(self, idx, bs, seed, balanced=False):
        idx = np.asarray(idx, dtype=np.int64)
        self.rng = np.random.default_rng(int(seed)); self.bs = int(bs)
        pools = [idx[y_all[idx] == c] for c in (0, 1)] if balanced else [idx]
        self.pools = [p for p in pools if len(p)]
        assert self.pools, "Sampler rỗng"
        self.perm = [self.rng.permutation(p) for p in self.pools]; self.ptr = [0] * len(self.pools)
    def _take(self, k, n):
        out = []
        while n > 0:
            if self.ptr[k] >= len(self.perm[k]):
                self.perm[k] = self.rng.permutation(self.pools[k]); self.ptr[k] = 0
            m = min(n, len(self.perm[k]) - self.ptr[k])
            out.append(self.perm[k][self.ptr[k]:self.ptr[k] + m]); self.ptr[k] += m; n -= m
        return np.concatenate(out)
    def next(self):
        if len(self.pools) == 1:
            return self._take(0, self.bs)
        h = self.bs // 2
        return np.concatenate([self._take(0, self.bs - h), self._take(1, h)])

def subsample(idx, n, seed):
    idx = np.asarray(idx)
    return idx if len(idx) <= n else np.sort(np.random.default_rng(seed).choice(idx, n, replace=False))

print(f"Dữ liệu frame: {DATA_GB:.2f} GB | thường trú GPU nếu <= {cfg.gpu_resident_max_gb} GB "
      f"(fp16 nếu <= {2 * cfg.gpu_resident_max_gb} GB)")

In [ ]:
# =========================================================
# 7.2 Lịch λ_p và LR + Hình / lambda and learning-rate schedules
# =========================================================
def grl_lambda(it, lam_max):
    p = it / max(1, TOTAL_ITERS - 1)
    return float(lam_max * (2.0 / (1.0 + math.exp(-cfg.lambda_gamma * p)) - 1.0))

def lr_factor(step):
    warm = max(1, int(cfg.warmup_frac * TOTAL_ITERS))
    if step < warm:
        return float(step + 1) / warm
    prog = min(1.0, (step - warm) / max(1, TOTAL_ITERS - warm))
    return 1e-3 + (1 - 1e-3) * 0.5 * (1 + math.cos(math.pi * prog))

_its = np.arange(TOTAL_ITERS)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for lm in sorted(set((cfg.lambda_max,) + tuple(cfg.lambda_grid))):
    ax[0].plot(_its / cfg.iters_per_epoch, [grl_lambda(i, lm) for i in _its],
               lw=2.2 if lm == cfg.lambda_max else 1, label=f"λ_max = {lm:g}" + (" (main)" if lm == cfg.lambda_max else ""))
ax[0].axvline(SELECT_FROM_EPOCH, color="k", ls=":", lw=1)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("λ_p"); ax[0].set_title("(a) GRL coefficient schedule"); ax[0].legend()
ax[1].plot(_its / cfg.iters_per_epoch, [cfg.lr * lr_factor(i) for i in _its], color=PAL["ok"])
ax[1].axvline(SELECT_FROM_EPOCH, color="k", ls=":", lw=1, label="earliest selectable epoch")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("learning rate"); ax[1].set_title("(b) Learning-rate schedule"); ax[1].legend()
savefig(fig, "schedules",
        f"Training schedules: (a) gradient-reversal coefficient λ_p = λ_max(2/(1+exp(−{cfg.lambda_gamma:g}p)) − 1) and "
        "(b) warm-up + cosine learning rate. Checkpoints are never selected before the dotted line.")

In [ ]:
# =========================================================
# 7.3 Suy luận, chỉ số, DEV và PAD / inference, metrics, DEV and PAD
# =========================================================
def amp_on(device):
    return bool(cfg.amp and device.type == "cuda")

@torch.no_grad()
def infer(model, idx, device, want_feat=False, bs=256):
    model.eval(); P, Fs = [], []
    idx = np.asarray(idx)
    for s in range(0, len(idx), bs):
        xb = fetch(idx[s:s + bs], device)
        with amp_autocast(device, amp_on(device)):
            logit, f = model(xb, return_feat=True)
        P.append(torch.sigmoid(logit.float()).clamp(0, 1).cpu().numpy())
        if want_feat:
            Fs.append(f.float().cpu().numpy())
    p = np.nan_to_num(np.concatenate(P) if P else np.array([]), nan=0.5)
    return (p, np.concatenate(Fs)) if want_feat else p

def frame_bce(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def rec_probs_from(idx, p):
    g = pd.DataFrame(dict(uid=uid_all[np.asarray(idx)], y=y_all[np.asarray(idx)], p=p))
    return g.groupby("uid").agg(y=("y", "first"), p=("p", "mean"))

def safe_auc(y, p):
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan

def youden(y, p):
    if len(np.unique(y)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y, p)
    thr = np.where(np.isfinite(thr), thr, 1.0)
    return float(np.clip(thr[int(np.argmax(tpr - fpr))], 0, 1))

def _lr_domain(Xs, Xt):
    X = np.vstack([Xs, Xt]); y = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=1.0, max_iter=500).fit(sc.transform(X), y)

def pad_score(Xs, Xt, seed=0, k=2):
    # Proxy A-distance = 2(1 - 2·err) của bộ phân lớp miền tuyến tính, CV k-fold trên mẫu cân bằng.
    n = min(len(Xs), len(Xt))
    if n < 10:
        return np.nan
    rng = np.random.default_rng(seed)
    X = np.vstack([Xs[rng.choice(len(Xs), n, replace=False)], Xt[rng.choice(len(Xt), n, replace=False)]])
    y = np.r_[np.zeros(n), np.ones(n)]; errs = []
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(C=1.0, max_iter=500).fit(sc.transform(X[tr]), y[tr])
        errs.append(np.mean(clf.predict(sc.transform(X[te])) != y[te]))
    return pad_from_err(float(np.mean(errs)))

def dev_risk(f_src_tr, f_src_va, loss_va, f_tgt):
    # Deep Embedded Validation (You et al., 2019): rủi ro nguồn-val có trọng số mật độ đích/nguồn,
    # cộng biến điều khiển để giảm phương sai. KHÔNG dùng nhãn đích. Thấp hơn = tốt hơn.
    try:
        sc, clf = _lr_domain(f_src_tr, f_tgt)
        q = clf.predict_proba(sc.transform(f_src_va))[:, 1].clip(1e-4, 1 - 1e-4)
        w = np.clip(q / (1 - q) * (len(f_src_tr) / len(f_tgt)), 0, 20)
        L = w * loss_va
        C = np.cov(np.vstack([L, w]))
        eta = -C[0, 1] / max(C[1, 1], 1e-12)
        return float(L.mean() + eta * w.mean() - eta)
    except Exception:
        return np.nan

def _snapshot(model):
    return {k: (v.detach().half() if v.is_floating_point() else v.detach()).cpu().clone()
            for k, v in model.state_dict().items()}

def load_state(model, state):
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in state.items()})
    return model

RULES = ("last", "src_val", "dev", "oracle")
print("Inference/metrics/DEV/PAD OK | quy tắc chọn:", RULES, "| chính:", cfg.select_rule)

In [ ]:
# =========================================================
# 7.4 Huấn luyện MỘT job (bài toán × seed × fold × phương pháp) / single-job training engine
# =========================================================
def job_key(j):
    return f"{j['variant']}|t{j['task']}|s{j['seed']}|f{j['fold']}|{j['method']}|sh{j.get('shots', 0)}"

def method_hp(job):
    hp = dict(lambda_max=cfg.lambda_max, fir_p=cfg.aug_fir_p, noise_p=cfg.aug_noise_p,
              shift_p=cfg.aug_shift_p, augment=cfg.augment, balanced=cfg.class_balanced_source)
    hp.update(job.get("hp", {}))
    if not hp["augment"]:
        hp["fir_p"] = hp["noise_p"] = hp["shift_p"] = 0.0
    return hp

def job_target_split(job):
    P = PARTS[(job["task"], job["seed"])]
    if job["fold"] is None:
        return P["tgt_all_idx"], P["tgt_all_idx"]
    return P["fold_idx"][job["fold"]]

def labelled_target(job, tgt_tr):
    shots = job.get("shots", -1 if job["method"] == "oracle" else 0)
    if shots == 0:
        return np.array([], dtype=np.int64)
    uids = pd.unique(uid_all[np.asarray(tgt_tr)])
    if shots > 0:
        rng = np.random.default_rng(1000 * job["seed"] + (job["fold"] or 0))
        pick = []
        for c in (0, 1):
            u = np.array([x for x in uids if REC_LAB[x] == c])
            pick += list(rng.choice(u, min(shots, len(u)), replace=False))
        uids = pick
    return frames_of(uids)

def train_job(job, device, verbose=False):
    t0 = time.time()
    P = PARTS[(job["task"], job["seed"])]
    method, hp = job["method"], method_hp(job)
    tgt_tr, tgt_te = job_target_split(job)
    lab_idx = labelled_target(job, tgt_tr)
    use_tgt = method in ADAPT_METHODS
    src_tr, src_va = P["src_tr_idx"], P["src_va_idx"]
    seed = 100 * job["seed"] + (0 if job["fold"] is None else job["fold"] + 1) + 7 * job.get("seed_offset", 0)

    with _BUILD_LOCK:
        set_seed(seed)
        model = LeakNet().to(device)
        disc = None
        if method in ADV_METHODS:
            disc = DomainDisc(cfg.bottleneck_dim * (2 if method.startswith("cdan") else 1)).to(device)
    if hasattr(model, "specaug"):
        model.specaug.generator = torch.Generator().manual_seed(seed * 7919 + 13)
    g = torch.Generator(device=device); g.manual_seed(seed + 1)
    Y = torch.as_tensor(y_all, device=device)
    s_samp = Sampler(src_tr, cfg.batch_size, seed + 2, balanced=hp["balanced"])
    t_samp = Sampler(tgt_tr, cfg.batch_size, seed + 3) if use_tgt else None
    l_samp = Sampler(lab_idx, max(8, cfg.batch_size // 2), seed + 4, balanced=True) if len(lab_idx) else None
    pw = None
    if not hp["balanced"]:
        pos = float(y_all[src_tr].sum()); pw = torch.tensor([(len(src_tr) - pos) / max(pos, 1.0)], device=device)

    groups = [{"params": model.parameters(), "lr": cfg.lr}]
    if disc is not None:
        groups.append({"params": disc.parameters(), "lr": cfg.lr * cfg.disc_lr_mult})
    opt = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = make_grad_scaler(amp_on(device))
    all_params = list(model.parameters()) + (list(disc.parameters()) if disc is not None else [])
    eps = cfg.label_smoothing

    ep_rows, hist_rows, preds, snaps = [], [], {}, {}
    it, n_skip = 0, 0
    for ep in range(cfg.epochs):
        model.train()
        if disc is not None:
            disc.train()
        acc = defaultdict(float); nb = 0
        for _ in range(cfg.iters_per_epoch):
            lam = grl_lambda(it, hp["lambda_max"])
            i_s = s_samp.next()
            parts = [fetch(i_s, device, hp["augment"], g, hp)]
            ns, nt = len(i_s), 0
            if use_tgt:
                i_t = t_samp.next(); parts.append(fetch(i_t, device, hp["augment"], g, hp)); nt = len(i_t)
            y_lab = Y[torch.as_tensor(i_s, device=device)]
            if l_samp is not None:
                i_l = l_samp.next(); parts.append(fetch(i_l, device, hp["augment"], g, hp))
                y_lab = torch.cat([y_lab, Y[torch.as_tensor(i_l, device=device)]])
            x = torch.cat(parts)
            with amp_autocast(device, amp_on(device)):
                logit, f = model(x, return_feat=True)
                logit_lab = torch.cat([logit[:ns], logit[ns + nt:]])
                loss_cls = F.binary_cross_entropy_with_logits(logit_lab.float(), y_lab * (1 - eps) + eps / 2,
                                                              pos_weight=pw)
                loss_da = torch.zeros((), device=device); d_acc = np.nan
                if method in ADV_METHODS:
                    dy = torch.cat([torch.ones(ns, device=device), torch.zeros(nt, device=device)])
                    h = f[:ns + nt] if method == "dann" else cdan_input(f[:ns + nt], logit[:ns + nt])
                    d = disc(h, lam).float()
                    if method == "cdan_e":
                        w = entropy_weights(logit[:ns], logit[ns:ns + nt])
                        loss_da = (w * F.binary_cross_entropy_with_logits(d, dy, reduction="none")).sum() / w.sum()
                    else:
                        loss_da = F.binary_cross_entropy_with_logits(d, dy)
                    d_acc = float(((d > 0).float() == dy).float().mean())
                elif method == "coral":
                    loss_da = grl_lambda(it, cfg.coral_weight) * coral_loss(f[:ns], f[ns:ns + nt])
                elif method == "mmd":
                    loss_da = grl_lambda(it, cfg.mmd_weight) * mmd_loss(f[:ns], f[ns:ns + nt])
                loss = loss_cls + loss_da
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                n_skip += 1; it += 1; continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(all_params, cfg.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            acc["loss_cls"] += float(loss_cls); acc["loss_da"] += float(loss_da)
            if np.isfinite(d_acc):
                acc["disc_acc"] += d_acc
            nb += 1; it += 1
        ep_rows.append(dict(epoch=ep + 1, it=it, lam=grl_lambda(max(0, it - 1), hp["lambda_max"]),
                            lr=opt.param_groups[0]["lr"], loss_cls=acc["loss_cls"] / max(nb, 1),
                            loss_da=acc["loss_da"] / max(nb, 1),
                            disc_acc=(acc["disc_acc"] / max(nb, 1)) if method in ADV_METHODS else np.nan))

        if (ep + 1) % cfg.eval_every == 0 or ep == cfg.epochs - 1:
            p_va, f_va = infer(model, src_va, device, want_feat=True)
            l_va = frame_bce(y_all[src_va], p_va)
            g_va = rec_probs_from(src_va, p_va)
            p_te = infer(model, tgt_te, device)
            i_tu = subsample(tgt_tr, cfg.dev_max, seed + ep)
            p_tu, f_tu = infer(model, i_tu, device, want_feat=True)
            _, f_su = infer(model, subsample(src_tr, cfg.dev_max, seed + ep + 1), device, want_feat=True)
            g_te = rec_probs_from(tgt_te, p_te)
            hist_rows.append(dict(
                epoch=ep + 1, src_val_bce=float(l_va.mean()), src_val_auc=safe_auc(y_all[src_va], p_va),
                src_val_rec_auc=safe_auc(g_va.y, g_va.p), thr_src=youden(g_va.y.values, g_va.p.values),
                tgt_entropy=float(np.mean(frame_bce(p_tu, p_tu))) if len(p_tu) else np.nan,
                dev=dev_risk(f_su, f_va, l_va, f_tu), pad=pad_score(f_su, f_tu, seed=ep),
                # --- chỉ để GIÁM SÁT / quy tắc oracle (dùng nhãn đích) ---
                tgt_auc_frame=safe_auc(y_all[tgt_te], p_te), tgt_auc_rec=safe_auc(g_te.y, g_te.p)))
            preds[ep + 1] = p_te.astype(np.float16)
            if job.get("keep_state"):
                snaps[ep + 1] = _snapshot(model)
            if verbose:
                h = hist_rows[-1]
                print(f"   ep{ep + 1:03d} cls {ep_rows[-1]['loss_cls']:.3f} da {ep_rows[-1]['loss_da']:.3f} "
                      f"dacc {ep_rows[-1]['disc_acc']:.2f} | srcval auc {h['src_val_auc']:.3f} | PAD {h['pad']:.2f} "
                      f"| [giám sát] tgt rec-AUC {h['tgt_auc_rec']:.3f}")

    H = pd.DataFrame(hist_rows)
    el = H[H.epoch >= SELECT_FROM_EPOCH]
    el = el if len(el) else H
    last = int(el.epoch.iloc[-1])
    def _arg(col, fn):
        v = el[col]
        return int(el.loc[getattr(v, fn)(), "epoch"]) if v.notna().any() else last
    sel = dict(last=last, src_val=_arg("src_val_bce", "idxmin"), dev=_arg("dev", "idxmin"),
               oracle=_arg("tgt_auc_rec", "idxmax"))
    Hs = H.set_index("epoch")
    res = dict(key=job_key(job), variant=job["variant"], task=job["task"], seed=job["seed"], fold=job["fold"],
               method=method, shots=job.get("shots", 0), hp=hp, hist=H, ep_hist=pd.DataFrame(ep_rows),
               te_idx=np.asarray(tgt_te), sel=sel,
               pred={r: preds[e].astype(np.float32) for r, e in sel.items()},
               thr={r: float(Hs.loc[e, "thr_src"]) for r, e in sel.items()},
               src_val_rec_auc={r: float(Hs.loc[e, "src_val_rec_auc"]) for r, e in sel.items()},
               state=snaps.get(sel[cfg.select_rule]) if job.get("keep_state") else None,
               n_skip=int(n_skip), time_s=time.time() - t0, device=str(device))
    del model, disc, snaps; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res

# ---- Thử nhanh một job để bắt lỗi sớm (2 epoch ngắn, không lưu) ----
_save = (cfg.epochs, cfg.iters_per_epoch, cfg.eval_every)
cfg.epochs, cfg.iters_per_epoch, cfg.eval_every = 2, 2, 1
TOTAL_ITERS = cfg.epochs * cfg.iters_per_epoch; _sfe = SELECT_FROM_EPOCH; SELECT_FROM_EPOCH = 1
_probe = train_job(dict(variant="probe", task=0, seed=cfg.seeds[0], fold=0, method="cdan_e"), DEVICE, verbose=True)
cfg.epochs, cfg.iters_per_epoch, cfg.eval_every = _save
TOTAL_ITERS = cfg.epochs * cfg.iters_per_epoch; SELECT_FROM_EPOCH = _sfe
print(f"✓ job thử CDAN+E chạy trọn ({_probe['time_s']:.1f}s, {len(_probe['te_idx'])} frame kiểm thử).")

In [ ]:
# =========================================================
# 7.5 Hàng đợi job trên các GPU + AdaBN + lưu/khôi phục / multi-GPU job queue, AdaBN, resume
# =========================================================
RES_PATH = OUT / "ada_results.pkl"
RUN_FP = hashlib.sha1((DATA_FP + json.dumps(_cfg_print, sort_keys=True, default=str)
                       + str(F_MAX_EFF)).encode()).hexdigest()[:12]
RESULTS = {}
if cfg.resume and RES_PATH.exists():
    try:
        _blob = pickle.load(open(RES_PATH, "rb"))
        if _blob.get("fp") == RUN_FP:
            RESULTS = _blob["results"]; print(f"Khôi phục {len(RESULTS)} job đã chạy (cùng cấu hình {RUN_FP}).")
        else:
            print("Kết quả cũ thuộc cấu hình khác -> chạy lại từ đầu.")
    except Exception as e:
        print("Không đọc được kết quả cũ:", type(e).__name__)

_SAVE_LOCK = threading.Lock()
def save_results():
    with _SAVE_LOCK:
        tmp = RES_PATH.with_suffix(".tmp")
        pickle.dump(dict(fp=RUN_FP, results=RESULTS), open(tmp, "wb"))
        os.replace(tmp, RES_PATH)

def run_jobs(jobs, title=""):
    jobs = [j for j in jobs if job_key(j) not in RESULTS]
    if not jobs:
        print(f"[{title}] mọi job đã có kết quả."); return
    ngpu = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
    devices = ([torch.device(f"cuda:{i}") for i in range(min(ngpu, cfg.max_parallel))]
               if (cfg.parallel_jobs and ngpu > 1) else [DEVICE])
    print(f"[{title}] {len(jobs)} job trên {[str(d) for d in devices]}")
    q = queue.Queue()
    for j in jobs:
        q.put(j)
    lock, done, t0, failed = threading.Lock(), [0], time.time(), []

    def worker(dev):
        while True:
            try:
                j = q.get_nowait()
            except queue.Empty:
                return
            try:
                r = train_job(j, dev)
                with lock:
                    RESULTS[r["key"]] = r; done[0] += 1
                    el = time.time() - t0; eta = el / done[0] * (len(jobs) - done[0])
                    h = r["hist"].set_index("epoch").loc[r["sel"][cfg.select_rule]]
                    print(f"  [{done[0]:3d}/{len(jobs)}] {j['variant']:>10s} {TASKS[j['task']]['name']:>18s} "
                          f"s{j['seed']} f{j['fold']} {j['method']:<11s} {r['time_s']:5.0f}s @{dev} | "
                          f"srcval AUC {h.src_val_auc:.3f} | [giám sát] tgt rec-AUC {h.tgt_auc_rec:.3f} | ETA {eta / 60:.0f} ph")
                    if done[0] % 10 == 0:
                        save_results()
            except Exception as e:
                with lock:
                    failed.append(j); print(f"  !! {job_key(j)} lỗi trên {dev}: {type(e).__name__}: {e}")
                    traceback.print_exc()

    ths = [threading.Thread(target=worker, args=(d,), daemon=True) for d in devices]
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    for j in failed:                                   # chạy lại tuần tự một lần
        r = train_job(j, devices[0]); RESULTS[r["key"]] = r
    save_results()
    print(f"[{title}] xong trong {(time.time() - t0) / 60:.1f} phút")

def adabn_model(state, tgt_tr, device):
    # AdaBN (Li et al., 2018): giữ nguyên trọng số source-only, ƯỚC LƯỢNG LẠI thống kê BatchNorm
    # bằng dữ liệu đích không nhãn (trung bình tích luỹ, không augmentation).
    m = load_state(LeakNet().to(device), state)
    bns = [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]
    for b in bns:
        b.reset_running_stats(); b.momentum = None
    m.eval()
    for b in bns:
        b.train()
    with torch.no_grad():
        idx = np.random.default_rng(0).permutation(np.asarray(tgt_tr))
        for s in range(0, len(idx), 128):
            with amp_autocast(device, amp_on(device)):
                m(fetch(idx[s:s + 128], device))
    return m.eval()

def run_adabn(variant="main"):
    n = 0
    for T in TASKS:
        for s in cfg.seeds:
            src = RESULTS.get(job_key(dict(variant=variant, task=T["i"], seed=s, fold=None, method="source_only")))
            if src is None or src["state"] is None:
                continue
            P = PARTS[(T["i"], s)]
            for k in range(len(P["folds"])):
                j = dict(variant=variant, task=T["i"], seed=s, fold=k, method="adabn")
                if job_key(j) in RESULTS:
                    continue
                t0 = time.time()
                tgt_tr, tgt_te = P["fold_idx"][k]
                m = adabn_model(src["state"], tgt_tr, DEVICE)
                p = infer(m, tgt_te, DEVICE); del m
                RESULTS[job_key(j)] = dict(key=job_key(j), variant=variant, task=T["i"], seed=s, fold=k,
                                           method="adabn", shots=0, hp={}, hist=None, ep_hist=None,
                                           te_idx=np.asarray(tgt_te), sel=dict(src["sel"]),
                                           pred={r: p for r in RULES}, thr=dict(src["thr"]),
                                           src_val_rec_auc=dict(src["src_val_rec_auc"]),
                                           state=None, n_skip=0, time_s=time.time() - t0, device=str(DEVICE))
                n += 1
    save_results()
    print(f"AdaBN: {n} lượt ước lượng lại BatchNorm.")
print("Job runner OK")

## 8. Thí nghiệm chính / Main experiment

`bài toán × seed × fold × phương pháp`. `source_only` chạy một lần mỗi seed (giữ trọng số cho AdaBN), các phương pháp
còn lại chạy theo từng fold đích. Trọng số (seed đầu, fold 0) được giữ cho phân tích không gian đặc trưng ở mục 11.

In [ ]:
# =========================================================
# 8.1 Danh sách job + chạy / build job list and run
# =========================================================
TRAINED = [m for m in cfg.methods if m in ("coral", "mmd", "dann", "cdan", "cdan_e", "oracle")]
NEED_SO = ("source_only" in cfg.methods) or ("adabn" in cfg.methods)

def main_jobs(variant="main", methods=None, seeds=None, tasks=None, hp=None, shots=0):
    methods = TRAINED if methods is None else methods
    seeds = cfg.seeds if seeds is None else seeds
    tasks = [T["i"] for T in TASKS] if tasks is None else tasks
    out = []
    for t in tasks:
        for s in seeds:
            for k in range(len(PARTS[(t, s)]["folds"])):
                for m in methods:
                    j = dict(variant=variant, task=t, seed=s, fold=k, method=m,
                             keep_state=(variant == "main" and s == cfg.seeds[0] and k == 0))
                    if hp:
                        j["hp"] = hp
                    if shots:
                        j["shots"] = shots
                    out.append(j)
    return out

MAIN_JOBS = []
if NEED_SO:
    MAIN_JOBS += [dict(variant="main", task=T["i"], seed=s, fold=None, method="source_only", keep_state=True)
                  for T in TASKS for s in cfg.seeds]
MAIN_JOBS += main_jobs()
print(f"Thí nghiệm chính: {len(MAIN_JOBS)} job huấn luyện "
      f"({len(TASKS)} bài toán × {len(cfg.seeds)} seed; phương pháp {list(cfg.methods)})")
run_jobs(MAIN_JOBS, "main")
if "adabn" in cfg.methods:
    run_adabn("main")

if cfg.save_models:
    for r in RESULTS.values():
        if r["variant"] == "main" and r.get("state") is not None:
            torch.save(r["state"], CKPT_DIR / f"{TASKS[r['task']]['slug']}_{r['method']}_s{r['seed']}_f{r['fold']}.pt")

_tm = pd.DataFrame([dict(method=r["method"], time_s=r["time_s"], device=r["device"], n_skip=r["n_skip"])
                    for r in RESULTS.values() if r["variant"] == "main"])
cost = _tm.groupby("method").agg(n_jobs=("time_s", "size"), mean_s=("time_s", "mean"),
                                 total_min=("time_s", lambda s: s.sum() / 60),
                                 skipped_steps=("n_skip", "sum")).reset_index()
save_table(cost, "training_cost", "Training cost per method (wall-clock per job and total; skipped non-finite AMP steps).", 1)

## 9. Kết quả trên miền đích / Target-domain results (RQ2)

**Ước lượng chính** (giống v7): xác suất OOF của từng frame đích được **trung bình qua các seed**, gộp thành xác suất bản ghi
bằng `p_mean`, rồi tính AUC mức bản ghi; CI 95% bằng bootstrap **bản ghi** (`cfg.bootstrap_n` lần). Cột `AUC (seeds)` là
trung bình ± độ lệch chuẩn giữa các seed (độ bất định do phân hoạch + khởi tạo). ΔAUC so với *source-only* là bootstrap
**ghép cặp** trên cùng các bản ghi; p hiệu chỉnh **Holm** trong mỗi bài toán. *Gap closed* = (AUC − AUC<sub>SO</sub>) /
(AUC<sub>oracle</sub> − AUC<sub>SO</sub>).

In [ ]:
# =========================================================
# 9.1 Công cụ tổng hợp + bootstrap / aggregation and bootstrap helpers
# =========================================================
def results_where(variant="main", task=None, method=None, seed=None, shots=None):
    return [r for r in RESULTS.values()
            if r["variant"] == variant and (task is None or r["task"] == task)
            and (method is None or r["method"] == method) and (seed is None or r["seed"] == seed)
            and (shots is None or r.get("shots", 0) == shots)]

def oof_vec(rs, rule):
    v = np.full(N_FRAMES, np.nan, dtype=np.float32)
    for r in rs:
        v[r["te_idx"]] = r["pred"][rule]
    return v

def rec_table(v):
    m = ~np.isnan(v)
    return (pd.DataFrame(dict(uid=uid_all[m], y=y_all[m], p=v[m]))
            .groupby("uid").agg(y=("y", "first"), p=("p", "mean")))

def rec_metrics(g, thr=None):
    thr = cfg.threshold if thr is None else thr
    y = g.y.values.astype(int); p = g.p.values; yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    _, _, f1, _ = precision_recall_fscore_support(y, yh, average="binary", zero_division=0)
    return dict(auc=safe_auc(y, p), bacc=balanced_accuracy_score(y, yh), f1=float(f1),
                mcc=float(matthews_corrcoef(y, yh)), sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1),
                acc=(tp + tn) / max(len(y), 1), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), n=int(len(y)))

def boot_auc_ci(y, p, n=None, seed=0):
    n = n or cfg.bootstrap_n; y = np.asarray(y); p = np.asarray(p)
    rng = np.random.default_rng(seed); v = []
    for _ in range(int(n)):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            v.append(roc_auc_score(y[i], p[i]))
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if v else (np.nan, np.nan)

def paired_delta(y, pa, pb, n=None, seed=0):
    # Bootstrap GHÉP CẶP: lấy mẫu lại bản ghi, cùng chỉ số cho hai phương pháp.
    n = n or cfg.bootstrap_n; y = np.asarray(y); pa = np.asarray(pa); pb = np.asarray(pb)
    d0 = safe_auc(y, pa) - safe_auc(y, pb)
    rng = np.random.default_rng(seed); v = []
    for _ in range(int(n)):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            v.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    if not v:
        return d0, np.nan, np.nan, np.nan
    v = np.asarray(v)
    p = float(min(1.0, 2 * min(np.mean(v <= 0), np.mean(v >= 0))))
    return d0, float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), p

def holm(p):
    p = np.asarray(p, dtype=float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def mcnemar(y, a, b):
    ca, cb = (np.asarray(a) == y), (np.asarray(b) == y)
    n01, n10 = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
    if n01 + n10 == 0:
        return n01, n10, 1.0
    return n01, n10, float(spstats.binomtest(min(n01, n10), n01 + n10, .5).pvalue)

def method_summary(task, method, variant="main", rule=None, seeds=None, shots=None):
    rule = rule or cfg.select_rule; seeds = cfg.seeds if seeds is None else seeds
    vs, per, srcv, thr = [], [], [], []
    for s in seeds:
        rs = results_where(variant, task, method, s, shots)
        if not rs:
            continue
        v = oof_vec(rs, rule); vs.append(v)
        per.append(dict(seed=s, **rec_metrics(rec_table(v))))
        srcv += [r["src_val_rec_auc"][rule] for r in rs]
        thr += [r["thr"][rule] for r in rs]
    if not vs:
        return None
    with np.errstate(all="ignore"):
        V = np.nanmean(np.vstack(vs), axis=0) if len(vs) > 1 else vs[0]
    g = rec_table(V)
    # Ngưỡng Youden trên src_val (không dùng nhãn đích) -> phân tích độ nhạy của ngưỡng cố định 0.5
    return dict(g=g, V=V, per=pd.DataFrame(per), ens=rec_metrics(g), src_val=float(np.nanmean(srcv)),
                thr_src=float(np.nanmean(thr)), ens_srcthr=rec_metrics(g, float(np.nanmean(thr))))

METHODS_REPORT = [m for m in ("source_only", "adabn", "coral", "mmd", "dann", "cdan", "cdan_e", "oracle")
                  if m in cfg.methods]
print("Phương pháp báo cáo:", [mlabel(m) for m in METHODS_REPORT])

In [ ]:
# =========================================================
# 9.2 BẢNG KẾT QUẢ CHÍNH / Main results table
# =========================================================
def build_main_table(variant="main", rule=None, seeds=None, methods=None, verbose=True):
    rule = rule or cfg.select_rule; methods = methods or METHODS_REPORT
    rows, store = [], {}
    for T in TASKS:
        tgt_uids = set(records.loc[records.domain == T["tgt"], "record_uid"])
        ref = method_summary(T["i"], "source_only", variant, rule, seeds)
        orc = method_summary(T["i"], "oracle", variant, rule, seeds)
        fam = []
        for m in methods:
            S = method_summary(T["i"], m, variant, rule, seeds)
            if S is None:
                continue
            if verbose and set(S["g"].index) != tgt_uids:
                print(f"  !! {T['name']} {m}: phủ {len(S['g'])}/{len(tgt_uids)} bản ghi đích")
            e = S["ens"]; lo, hi = boot_auc_ci(S["g"].y.values, S["g"].p.values, seed=T["i"])
            mk = ~np.isnan(S["V"])
            row = dict(task=T["name"], method_key=m, method=mlabel(m), n_rec=e["n"], AUC=e["auc"], AUC_lo=lo,
                       AUC_hi=hi, AUC_seed_mean=float(S["per"].auc.mean()),
                       AUC_seed_sd=float(S["per"].auc.std(ddof=1)) if len(S["per"]) > 1 else np.nan,
                       BAcc=e["bacc"], F1=e["f1"], MCC=e["mcc"], Sens=e["sens"], Spec=e["spec"],
                       BAcc_srcthr=S["ens_srcthr"]["bacc"], thr_src=S["thr_src"],
                       AUC_frame=safe_auc(y_all[mk], S["V"][mk]), SrcVal_AUC=S["src_val"],
                       dAUC=np.nan, dAUC_lo=np.nan, dAUC_hi=np.nan, p_boot=np.nan, p_holm=np.nan,
                       gap_closed_pct=np.nan)
            if ref is not None and m != "source_only":
                common = S["g"].index.intersection(ref["g"].index)
                d, dlo, dhi, p = paired_delta(S["g"].loc[common, "y"].values, S["g"].loc[common, "p"].values,
                                              ref["g"].loc[common, "p"].values, seed=17 + T["i"])
                row.update(dAUC=d, dAUC_lo=dlo, dAUC_hi=dhi, p_boot=p)
                fam.append(len(rows))
            if ref is not None and orc is not None and m not in ("source_only", "oracle"):
                gap = orc["ens"]["auc"] - ref["ens"]["auc"]
                if gap > 0.02:
                    row["gap_closed_pct"] = 100 * (e["auc"] - ref["ens"]["auc"]) / gap
            rows.append(row); store[(T["i"], m)] = S
        if fam:
            adj = holm([rows[i]["p_boot"] for i in fam])
            for i, a in zip(fam, adj):
                rows[i]["p_holm"] = float(a)
    return pd.DataFrame(rows), store

MAIN, MAIN_STORE = build_main_table()

def _ci(a, lo, hi, nd=3):
    return "–" if not np.isfinite(a) else f"{a:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

pretty = pd.DataFrame(dict(
    Task=MAIN.task, Method=MAIN.method,
    **{"AUC [95% CI]": [_ci(a, l, h) for a, l, h in zip(MAIN.AUC, MAIN.AUC_lo, MAIN.AUC_hi)],
       "AUC seeds (mean ± sd)": [f"{a:.3f} ± {s:.3f}" if np.isfinite(s) else f"{a:.3f}"
                                 for a, s in zip(MAIN.AUC_seed_mean, MAIN.AUC_seed_sd)],
       "BAcc": MAIN.BAcc.round(3), "BAcc@src-thr": MAIN.BAcc_srcthr.round(3),
       "F1": MAIN.F1.round(3), "MCC": MAIN.MCC.round(3),
       "ΔAUC vs SO [95% CI]": [_ci(a, l, h) for a, l, h in zip(MAIN.dAUC, MAIN.dAUC_lo, MAIN.dAUC_hi)],
       "p (Holm)": MAIN.p_holm.round(4), "Gap closed (%)": MAIN.gap_closed_pct.round(1),
       "Source-val AUC": MAIN.SrcVal_AUC.round(3)}))
MAIN.to_csv(TAB_DIR / "main_results_numeric.csv", index=False)
save_table(pretty, "main_results",
           f"Target-domain recording-level results (checkpoint rule '{cfg.select_rule}', {len(cfg.seeds)} seeds, "
           f"{cfg.record_agg} aggregation, threshold {cfg.threshold}). ΔAUC: paired bootstrap over target recordings vs "
           "source-only, Holm-adjusted within task. BAcc@src-thr: balanced accuracy at the Youden threshold of the "
           "source-validation recordings (label-free on target). Source-val AUC: retained performance on held-out "
           "source recordings.", 3)

In [ ]:
# =========================================================
# 9.3 Hình: AUC đích theo phương pháp + ΔAUC (forest) / target AUC and paired ΔAUC
# =========================================================
nT = len(TASKS)
fig, axes = plt.subplots(nT, 2, figsize=(13, 0.55 * len(METHODS_REPORT) * nT + 1.5), squeeze=False)
for ti, T in enumerate(TASKS):
    d = MAIN[MAIN.task == T["name"]].reset_index(drop=True)
    ax = axes[ti, 0]
    for i, r in d.iterrows():
        c = METHOD_COLOR.get(r.method_key, "k")
        ax.errorbar(r.AUC, i, xerr=[[r.AUC - r.AUC_lo], [r.AUC_hi - r.AUC]], fmt="o", color=c, capsize=3, ms=7)
        per = MAIN_STORE[(T["i"], r.method_key)]["per"].auc.values
        ax.scatter(per, np.full(len(per), i) + .22, s=12, color=c, alpha=.6, marker="|")
    so = d.loc[d.method_key == "source_only", "AUC"]
    if len(so):
        ax.axvline(float(so.iloc[0]), color=METHOD_COLOR["source_only"], ls="--", lw=1)
    ax.axvline(.5, color="k", lw=.6, ls=":")
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.method); ax.invert_yaxis()
    ax.set_xlabel("target record-level AUC"); ax.set_title(f"({'abcdefgh'[2 * ti]}) {T['name']}: AUC [95% CI], | = seeds")
    ax = axes[ti, 1]
    dd = d[d.method_key != "source_only"].reset_index(drop=True)
    for i, r in dd.iterrows():
        sig = np.isfinite(r.p_holm) and r.p_holm < .05
        ax.errorbar(r.dAUC, i, xerr=[[r.dAUC - r.dAUC_lo], [r.dAUC_hi - r.dAUC]], fmt="D" if sig else "o",
                    color=METHOD_COLOR.get(r.method_key, "k"), capsize=3, ms=6, mfc=None if sig else "white")
        ax.text(1.01, i, f"p={r.p_holm:.3f}" if np.isfinite(r.p_holm) else "", va="center", fontsize=8,
                transform=ax.get_yaxis_transform(), clip_on=False)
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(range(len(dd))); ax.set_yticklabels(dd.method); ax.invert_yaxis()
    ax.set_xlabel("ΔAUC vs source-only (paired bootstrap)")
    ax.set_title(f"({'abcdefgh'[2 * ti + 1]}) improvement over source-only (filled = Holm p < 0.05)")
savefig(fig, "target_auc_forest",
        "Target-domain recording-level AUC per method with 95% bootstrap CIs and per-seed values (left), and paired "
        "ΔAUC relative to source-only with Holm-adjusted p-values (right). Dashed line: source-only AUC.")

if nT >= 2:
    H_ = MAIN.pivot(index="method", columns="task", values="AUC").reindex([mlabel(m) for m in METHODS_REPORT])
    fig, ax = plt.subplots(figsize=(1.6 + 1.8 * nT, 0.45 * len(H_) + 1.2))
    im = ax.imshow(H_.values, cmap="viridis", vmin=max(.5, np.nanmin(H_.values) - .02), vmax=1)
    for i in range(H_.shape[0]):
        for j in range(H_.shape[1]):
            ax.text(j, i, f"{H_.values[i, j]:.3f}", ha="center", va="center",
                    color="w" if H_.values[i, j] < np.nanmean(H_.values) else "k", fontsize=9)
    ax.set_xticks(range(H_.shape[1])); ax.set_xticklabels(H_.columns, rotation=15)
    ax.set_yticks(range(H_.shape[0])); ax.set_yticklabels(H_.index); ax.grid(False)
    plt.colorbar(im, ax=ax, fraction=.046, label="target AUC")
    savefig(fig, "auc_heatmap_tasks", "Target record-level AUC of every method on every transfer task.")

In [ ]:
# =========================================================
# 9.4 ROC mức bản ghi + ma trận nhầm lẫn / record-level ROC curves and confusion matrices
# =========================================================
fig, axes = plt.subplots(1, nT, figsize=(5.2 * nT, 4.6), squeeze=False)
for ti, T in enumerate(TASKS):
    ax = axes[0, ti]
    for m in METHODS_REPORT:
        S = MAIN_STORE.get((T["i"], m))
        if S is None:
            continue
        fpr, tpr, _ = roc_curve(S["g"].y, S["g"].p)
        ax.plot(fpr, tpr, color=METHOD_COLOR.get(m, "k"), lw=2.2 if m in ADV_METHODS else 1.3,
                ls="--" if m in ("source_only", "oracle") else "-", label=f"{mlabel(m)} ({S['ens']['auc']:.3f})")
    ax.plot([0, 1], [0, 1], color="k", lw=.6, ls=":")
    ax.set_xlabel("false-positive rate (recordings)"); ax.set_ylabel("true-positive rate")
    ax.set_title(T["name"]); ax.legend(fontsize=8, loc="lower right")
savefig(fig, "roc_target", "Record-level ROC curves on the target domain (seed-averaged out-of-fold probabilities).")

fig, axes = plt.subplots(nT, len(METHODS_REPORT), figsize=(2.3 * len(METHODS_REPORT), 2.4 * nT), squeeze=False)
for ti, T in enumerate(TASKS):
    for mi, m in enumerate(METHODS_REPORT):
        ax = axes[ti, mi]; S = MAIN_STORE.get((T["i"], m))
        if S is None:
            ax.axis("off"); continue
        e = S["ens"]; C = np.array([[e["tn"], e["fp"]], [e["fn"], e["tp"]]])
        ax.imshow(C, cmap="Blues", vmin=0, vmax=max(1, C.max()))
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(C[i, j]), ha="center", va="center",
                        color="w" if C[i, j] > C.max() / 2 else "k", fontsize=11)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["noleak", "leak"], fontsize=7)
        ax.set_yticks([0, 1]); ax.set_yticklabels(["noleak", "leak"], fontsize=7); ax.grid(False)
        ax.set_title(f"{mlabel(m)}\nBAcc {e['bacc']:.2f}", fontsize=8)
        if mi == 0:
            ax.set_ylabel(f"{T['name']}\ntrue", fontsize=8)
savefig(fig, "confusion_target",
        f"Record-level confusion matrices on the target domain at the pre-registered threshold {cfg.threshold} "
        "(rows: true class; columns: predicted).")

# McNemar mức bản ghi vs source-only (bổ sung cho ΔAUC; dùng quyết định ở ngưỡng cố định)
mc = []
for T in TASKS:
    ref = MAIN_STORE.get((T["i"], "source_only"))
    if ref is None:
        continue
    for m in METHODS_REPORT:
        S = MAIN_STORE.get((T["i"], m))
        if S is None or m == "source_only":
            continue
        common = S["g"].index.intersection(ref["g"].index)
        y = S["g"].loc[common, "y"].values.astype(int)
        n01, n10, p = mcnemar(y, (S["g"].loc[common, "p"].values >= cfg.threshold).astype(int),
                              (ref["g"].loc[common, "p"].values >= cfg.threshold).astype(int))
        mc.append(dict(task=T["name"], method=mlabel(m), only_method_correct=n01, only_SO_correct=n10, p_exact=p))
if mc:
    MC = pd.DataFrame(mc)
    MC["p_holm"] = np.concatenate([holm(g.p_exact.values) for _, g in MC.groupby("task", sort=False)])
    save_table(MC, "mcnemar_vs_source_only",
               "Exact McNemar test on record-level decisions (threshold fixed in advance) of each method versus "
               "source-only; Holm correction within task.", 4)

## 10. Động học huấn luyện đối nghịch / Adversarial training dynamics (RQ3)

Dấu hiệu một lượt DANN "lành mạnh": độ chính xác của $G_d$ **giảm về ≈ 0.5** khi $\lambda_p$ tăng (đặc trưng hai miền
không còn phân biệt được), trong khi mất mát phân lớp nguồn vẫn giảm. PAD của đặc trưng (đo bằng bộ phân lớp tuyến tính
độc lập, mục 7.3) phải **thấp hơn** source-only. Đường AUC đích theo epoch dùng nhãn đích nên **chỉ để quan sát**.

In [ ]:
# =========================================================
# 10.1 Mất mát, độ chính xác bộ phân biệt miền, λ_p / losses, discriminator accuracy, lambda
# =========================================================
s0 = cfg.seeds[0]
adv_present = [m for m in ("dann", "cdan", "cdan_e") if m in cfg.methods]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for m in [x for x in ("source_only", "coral", "mmd", "dann", "cdan_e", "oracle") if x in cfg.methods]:
    rs = [r for r in results_where("main", 0, m, s0) if r.get("ep_hist") is not None]
    rs = [r for r in rs if r["fold"] in (0, None)]
    if not rs:
        continue
    E = rs[0]["ep_hist"]; c = METHOD_COLOR.get(m, "k")
    axes[0].plot(E.epoch, E.loss_cls, color=c, label=mlabel(m))
    if m in ADV_METHODS:
        axes[1].plot(E.epoch, E.loss_da, color=c, label=mlabel(m))
        axes[2].plot(E.epoch, E.disc_acc, color=c, label=mlabel(m))
    elif m in ("coral", "mmd"):
        axes[1].plot(E.epoch, E.loss_da, color=c, ls="--", label=f"{mlabel(m)} (weighted)")
    if m in ADV_METHODS or m == "coral":
        axes[3].plot(E.epoch, E.lam, color=c, label=mlabel(m))
axes[0].set_title("(a) label loss $\\mathcal{L}_y$ (train)"); axes[0].legend(fontsize=7)
axes[1].set_title("(b) domain loss $\\mathcal{L}_d$ / alignment loss"); axes[1].legend(fontsize=7)
axes[1].axhline(math.log(2), color="k", ls=":", lw=.8)
axes[2].set_title("(c) discriminator accuracy (train batches)"); axes[2].axhline(.5, color="k", ls=":", lw=.8)
axes[2].set_ylim(.3, 1.02); axes[2].legend(fontsize=7)
axes[3].set_title("(d) adaptation weight λ_p"); axes[3].legend(fontsize=7)
for a in axes:
    a.set_xlabel("epoch")
savefig(fig, "adversarial_dynamics",
        f"Training dynamics on {TASKS[0]['name']} (seed {s0}, fold 0): (a) source label loss, (b) domain-discriminator "
        "loss (dotted: ln 2, the chance level) or weighted CORAL/MMD loss, (c) discriminator accuracy, (d) λ_p.")

In [ ]:
# =========================================================
# 10.2 AUC đích [giám sát] và PAD theo epoch, trung bình qua fold × seed / monitored target AUC and PAD
# =========================================================
fig, axes = plt.subplots(nT, 3, figsize=(16, 3.4 * nT), squeeze=False)
for ti, T in enumerate(TASKS):
    for m in [x for x in METHODS_REPORT if x != "adabn"]:
        rs = [r for r in results_where("main", T["i"], m) if r.get("hist") is not None]
        if not rs:
            continue
        H = pd.concat([r["hist"].assign(job=r["key"]) for r in rs])
        agg = H.groupby("epoch").agg(a=("tgt_auc_rec", "mean"), sa=("tgt_auc_rec", "std"),
                                     pad=("pad", "mean"), spad=("pad", "std"), sv=("src_val_rec_auc", "mean"))
        c = METHOD_COLOR.get(m, "k")
        axes[ti, 0].plot(agg.index, agg.a, color=c, label=mlabel(m))
        axes[ti, 0].fill_between(agg.index, agg.a - agg.sa.fillna(0), agg.a + agg.sa.fillna(0), color=c, alpha=.12)
        axes[ti, 1].plot(agg.index, agg.pad, color=c, label=mlabel(m))
        axes[ti, 2].plot(agg.index, agg.sv, color=c, label=mlabel(m))
    for a in axes[ti]:
        a.axvline(SELECT_FROM_EPOCH, color="k", ls=":", lw=.8); a.set_xlabel("epoch")
    axes[ti, 0].set_title(f"{T['name']}: target rec-AUC [monitor only]"); axes[ti, 0].legend(fontsize=7)
    axes[ti, 0].set_ylim(-.02, 1.02); axes[ti, 2].set_ylim(-.02, 1.02)
    axes[ti, 1].set_title("proxy A-distance of f (source vs target)"); axes[ti, 1].set_ylim(-.05, 2.05)
    axes[ti, 2].set_title("source-validation rec-AUC")
savefig(fig, "epoch_curves",
        "Per-epoch behaviour averaged over folds and seeds (band: ±1 sd): target recording-level AUC (uses target labels, "
        "shown for monitoring only), proxy A-distance between source and target bottleneck features, and "
        "source-validation AUC. Dotted line: earliest selectable epoch.")

## 11. Không gian đặc trưng sau thích ứng / Feature-space analysis (RQ3)

Dùng các mô hình đã giữ (bài toán đầu, seed đầu, fold 0). Với mỗi phương pháp: **PAD** tổng (nguồn vs đích),
**PAD theo lớp** (căn chỉnh *có điều kiện*: leak-nguồn vs leak-đích, noleak-nguồn vs noleak-đích — dùng nhãn đích, chỉ để
phân tích), **MMD** Gauss, và **silhouette** theo lớp của đặc trưng đích (đặc trưng đích còn tách lớp không?).
Tham chiếu "input" = thống kê log-Mel (mục 3.7). Lý tưởng: PAD tổng giảm **và** silhouette lớp đích không giảm.

In [ ]:
# =========================================================
# 11.1 Mô hình đã giữ + đặc trưng + PAD/MMD/silhouette / kept models, features and discrepancy table
# =========================================================
FS_T, FS_S = 0, cfg.seeds[0]
P_fs = PARTS[(FS_T, FS_S)]
tgt_tr0, tgt_te0 = P_fs["fold_idx"][0]
KEPT = {}
_so = RESULTS.get(job_key(dict(variant="main", task=FS_T, seed=FS_S, fold=None, method="source_only")))
if _so is not None and _so["state"] is not None:
    KEPT["source_only"] = load_state(LeakNet().to(DEVICE), _so["state"]).eval()
    if "adabn" in cfg.methods:
        KEPT["adabn"] = adabn_model(_so["state"], tgt_tr0, DEVICE)
for m in TRAINED:
    r = RESULTS.get(job_key(dict(variant="main", task=FS_T, seed=FS_S, fold=0, method=m)))
    if r is not None and r["state"] is not None:
        KEPT[m] = load_state(LeakNet().to(DEVICE), r["state"]).eval()
KEPT = {m: KEPT[m] for m in METHODS_REPORT if m in KEPT}

_half = max(50, cfg.tsne_max_points // 2)
idx_S = subsample(np.concatenate([P_fs["src_tr_idx"], P_fs["src_va_idx"]]), _half, 11)
idx_T = subsample(P_fs["tgt_all_idx"], _half, 12)
yS, yT = y_all[idx_S], y_all[idx_T]

def gauss_mmd(A, B, n=600, seed=0):
    rng = np.random.default_rng(seed)
    A = A[rng.choice(len(A), min(n, len(A)), replace=False)]; B = B[rng.choice(len(B), min(n, len(B)), replace=False)]
    X = np.vstack([A, B]); X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    D = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
    K = np.exp(-D / (np.median(D[D > 0]) + 1e-9)); na = len(A)
    return float(K[:na, :na].mean() + K[na:, na:].mean() - 2 * K[:na, na:].mean())

FEATS, fs_rows = {}, []
FEATS["input (log-Mel stats)"] = (FEAT_IN[idx_S], FEAT_IN[idx_T])
for m, net in KEPT.items():
    FEATS[mlabel(m)] = (infer(net, idx_S, DEVICE, want_feat=True)[1], infer(net, idx_T, DEVICE, want_feat=True)[1])
for name, (A, B) in FEATS.items():
    sil = (silhouette_score(StandardScaler().fit_transform(B), yT)
           if len(np.unique(yT)) == 2 and len(B) > 10 else np.nan)
    fs_rows.append(dict(representation=name, PAD=pad_score(A, B, k=5),
                        PAD_leak=pad_score(A[yS == 1], B[yT == 1], k=5),
                        PAD_noleak=pad_score(A[yS == 0], B[yT == 0], k=5),
                        MMD=gauss_mmd(A, B), target_class_silhouette=sil))
FS_TAB = pd.DataFrame(fs_rows)
save_table(FS_TAB, "feature_discrepancy",
           f"Domain discrepancy of learned features on {TASKS[FS_T]['name']} (seed {FS_S}, fold-0 models; "
           f"{len(idx_S)} source and {len(idx_T)} target frames): proxy A-distance overall and per class, Gaussian MMD, "
           "and class silhouette of target features (higher = target classes better separated).", 3)

In [ ]:
# =========================================================
# 11.2 t-SNE trước/sau thích ứng + biểu đồ PAD / t-SNE before vs after adaptation, PAD bars
# =========================================================
names = list(FEATS.keys())
fig, axes = plt.subplots(2, len(names), figsize=(3.3 * len(names), 6.6), squeeze=False)
for j, name in enumerate(names):
    A, B = FEATS[name]
    Z = TSNE(2, init="pca", random_state=0,
             perplexity=min(30, max(5, (len(A) + len(B)) // 5))).fit_transform(
        StandardScaler().fit_transform(np.vstack([A, B])))
    za, zb = Z[:len(A)], Z[len(A):]
    ax = axes[0, j]
    ax.scatter(za[:, 0], za[:, 1], s=4, color=PAL["src"], alpha=.45, label="source")
    ax.scatter(zb[:, 0], zb[:, 1], s=4, color=PAL["tgt"], alpha=.45, label="target")
    pad = FS_TAB.loc[FS_TAB.representation == name, "PAD"].values[0]
    ax.set_title(f"{name}\nPAD = {pad:.2f}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    if j == 0:
        ax.legend(markerscale=3, fontsize=7); ax.set_ylabel("colour = domain")
    ax = axes[1, j]
    for lab, mk, c in [(0, "o", PAL["noleak"]), (1, "^", PAL["leak"])]:
        ax.scatter(za[yS == lab, 0], za[yS == lab, 1], s=4, marker=mk, color=c, alpha=.15)
        ax.scatter(zb[yT == lab, 0], zb[yT == lab, 1], s=7, marker=mk, color=c, alpha=.7,
                   label=f"target {'leak' if lab else 'noleak'}")
    sil = FS_TAB.loc[FS_TAB.representation == name, "target_class_silhouette"].values[0]
    ax.set_title(f"target silhouette = {sil:.2f}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    if j == 0:
        ax.legend(markerscale=2, fontsize=7); ax.set_ylabel("colour = class (faint = source)")
savefig(fig, "tsne_adaptation",
        f"t-SNE of bottleneck features on {TASKS[FS_T]['name']}: top row coloured by domain (source vs target), bottom row "
        "by class (target highlighted). Successful adaptation mixes the domains (lower PAD) while keeping target classes "
        "apart.")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
xx = np.arange(len(FS_TAB)); cols_ = ["#999999"] + [METHOD_COLOR.get(m, "k") for m in KEPT]
ax[0].bar(xx, FS_TAB.PAD, color=cols_[:len(FS_TAB)])
ax[0].set_xticks(xx); ax[0].set_xticklabels(FS_TAB.representation, rotation=25, ha="right", fontsize=8)
ax[0].set_ylabel("proxy A-distance"); ax[0].set_ylim(0, 2.05); ax[0].set_title("(a) Overall domain discrepancy")
w = .38
ax[1].bar(xx - w / 2, FS_TAB.PAD_noleak, w, color=PAL["noleak"], label="no-leak")
ax[1].bar(xx + w / 2, FS_TAB.PAD_leak, w, color=PAL["leak"], label="leak")
ax[1].set_xticks(xx); ax[1].set_xticklabels(FS_TAB.representation, rotation=25, ha="right", fontsize=8)
ax[1].set_ylim(0, 2.05); ax[1].set_title("(b) Class-conditional discrepancy"); ax[1].legend()
savefig(fig, "pad_bars",
        "Proxy A-distance between source and target features: (a) overall, (b) within each class. "
        "Class-conditional alignment (b) is what transfers the decision boundary.")

## 12. Chọn mô hình khi KHÔNG có nhãn đích / Model selection without target labels (RQ4)

| Quy tắc | Dùng gì | Hợp lệ cho UDA? |
|---|---|---|
| `last` | epoch cuối của lịch cố định | **có** (quy tắc chính, đăng ký trước) |
| `src_val` | BCE nhỏ nhất trên `src_val` | có |
| `dev` | rủi ro DEV nhỏ nhất (You et al., 2019) | có |
| `oracle` | AUC bản ghi **đích** lớn nhất | **không** — chỉ để đo mức lạc quan |

"Optimism" = AUC(`oracle`) − AUC(quy tắc chính): phần kết quả mà một bài báo sẽ thổi phồng nếu chọn epoch bằng nhãn đích.

In [ ]:
# =========================================================
# 12.1 AUC theo quy tắc chọn + tương quan tiêu chí với AUC đích / rule comparison and criterion validity
# =========================================================
sel_rows = []
for T in TASKS:
    for m in METHODS_REPORT:
        r_ = {}
        for rule in RULES:
            S = method_summary(T["i"], m, "main", rule)
            if S is not None:
                r_[rule] = S["ens"]["auc"]
        if r_:
            sel_rows.append(dict(task=T["name"], method=mlabel(m), **{f"AUC_{k}": v for k, v in r_.items()},
                                 optimism=r_.get("oracle", np.nan) - r_.get(cfg.select_rule, np.nan)))
SEL = pd.DataFrame(sel_rows)
save_table(SEL, "selection_rules",
           f"Target record-level AUC under different checkpoint-selection rules. 'oracle' uses target labels and is "
           f"reported only to quantify optimism (= AUC_oracle − AUC_{cfg.select_rule}).", 3)

corr = []
for m in [x for x in METHODS_REPORT if x != "adabn"]:
    for crit, sign in [("src_val_bce", -1), ("dev", -1), ("pad", -1), ("tgt_entropy", -1)]:
        rhos = []
        for r in results_where("main", None, m):
            H = r.get("hist")
            if H is None or len(H) < 4:
                continue
            a, b = sign * H[crit].values, H.tgt_auc_rec.values
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() >= 4 and np.std(a[ok]) > 0 and np.std(b[ok]) > 0:
                rhos.append(spstats.spearmanr(a[ok], b[ok]).correlation)
        if rhos:
            corr.append(dict(method=mlabel(m), criterion=f"−{crit}", n_runs=len(rhos),
                             spearman_mean=float(np.mean(rhos)), spearman_sd=float(np.std(rhos))))
if corr:
    CORR = pd.DataFrame(corr)
    save_table(CORR, "selection_criterion_validity",
               "Within-run Spearman correlation across evaluation epochs between each label-free criterion (sign-flipped "
               "so that higher is better) and the monitored target recording-level AUC. Values near zero mean the "
               "criterion cannot guide selection.", 3)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
d0 = SEL[SEL.task == TASKS[0]["name"]].reset_index(drop=True)
mk = {"last": "o", "src_val": "s", "dev": "D", "oracle": "*"}
dy_ = {"last": -.24, "src_val": -.08, "dev": .08, "oracle": .24}
for i, r in d0.iterrows():
    for rule in RULES:
        v = r.get(f"AUC_{rule}", np.nan)
        if np.isfinite(v):
            ax[0].scatter(v, i + dy_[rule], marker=mk[rule], s=90 if rule == "oracle" else 45,
                          color="#C8553D" if rule == "oracle" else "#2E6F95", alpha=.85,
                          label=rule if i == 0 else None)
ax[0].set_yticks(range(len(d0))); ax[0].set_yticklabels(d0.method); ax[0].invert_yaxis()
ax[0].set_xlabel("target record-level AUC"); ax[0].legend(fontsize=8)
ax[0].set_title(f"(a) {TASKS[0]['name']}: selection rule (★ uses target labels)")
_m = "dann" if "dann" in cfg.methods else (adv_present[0] if adv_present else None)
if _m:
    for r in results_where("main", 0, _m):
        H = r["hist"]
        ax[1].scatter(H.dev, H.tgt_auc_rec, s=12, alpha=.6)
    ax[1].set_xlabel("DEV risk (label-free, lower = better)"); ax[1].set_ylabel("target rec-AUC [monitor]")
    ax[1].set_title(f"(b) {mlabel(_m)}: DEV vs target AUC across epochs (colour = run)")
savefig(fig, "model_selection",
        "Model selection without target labels: (a) target AUC obtained with each checkpoint rule; (b) relationship "
        "between the label-free DEV criterion and the monitored target AUC across epochs and runs.")

## 13. Độ nhạy theo λ và ablation / λ sensitivity and ablation (RQ4)

Chạy trên bài toán `cfg.sweep_task`, các seed `cfg.sweep_seeds`, **mọi fold** đích. Mọi so sánh dùng **cùng seed** với lượt
chính. Biến thể ablation (DANN): bỏ FIR ngẫu nhiên (`no_fir`), bỏ mọi augmentation (`no_aug`), bỏ lấy mẫu nguồn cân bằng
lớp (`no_balance`); CDAN **không** trọng số entropy (`cdan_noent`) so với CDAN+E.

In [ ]:
# =========================================================
# 13.1 Quét λ_max cho DANN / lambda sweep
# =========================================================
SW_T, SW_S = cfg.sweep_task, tuple(cfg.sweep_seeds)
LAMBDAS = sorted(set((cfg.lambda_max,) + tuple(cfg.lambda_grid)))
if cfg.run_lambda_sweep and "dann" in cfg.methods:
    jobs = []
    for lm in LAMBDAS:
        if lm != cfg.lambda_max:
            jobs += main_jobs(variant=f"lam{lm:g}", methods=["dann"], seeds=SW_S, tasks=[SW_T], hp={"lambda_max": lm})
    run_jobs(jobs, "lambda sweep")
    ref = method_summary(SW_T, "source_only", "main", seeds=SW_S)
    orc = method_summary(SW_T, "oracle", "main", seeds=SW_S)
    lam_rows = []
    for lm in LAMBDAS:
        S = method_summary(SW_T, "dann", "main" if lm == cfg.lambda_max else f"lam{lm:g}", seeds=SW_S)
        if S is None:
            continue
        lo, hi = boot_auc_ci(S["g"].y, S["g"].p, seed=3)
        lam_rows.append(dict(lambda_max=lm, AUC=S["ens"]["auc"], lo=lo, hi=hi, BAcc=S["ens"]["bacc"],
                             src_val_AUC=S["src_val"]))
    LAM = pd.DataFrame(lam_rows)
    save_table(LAM, "lambda_sweep",
               f"Sensitivity of DANN to the final adversarial weight λ_max on {TASKS[SW_T]['name']} "
               f"(seeds {list(SW_S)}, all target folds).", 3)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.errorbar(LAM.lambda_max, LAM.AUC, yerr=[LAM.AUC - LAM.lo, LAM.hi - LAM.AUC], fmt="o-",
                color=METHOD_COLOR["dann"], capsize=4, label="DANN (target)")
    ax.plot(LAM.lambda_max, LAM.src_val_AUC, "s:", color="#555555", label="DANN (source-val)")
    if ref is not None:
        ax.axhline(ref["ens"]["auc"], color=METHOD_COLOR["source_only"], ls="--", label="source-only (target)")
    if orc is not None:
        ax.axhline(orc["ens"]["auc"], color=METHOD_COLOR["oracle"], ls="-.", label="oracle (target)")
    ax.set_xscale("log"); ax.set_xlabel("λ_max"); ax.set_ylabel("record-level AUC"); ax.legend(fontsize=8)
    ax.set_title(f"λ sensitivity — {TASKS[SW_T]['name']}")
    savefig(fig, "lambda_sensitivity",
            "Target AUC (95% CI) and source-validation AUC of DANN as a function of the final adversarial weight λ_max.")
else:
    print("Bỏ qua quét λ (cfg.run_lambda_sweep=False hoặc không có DANN).")

In [ ]:
# =========================================================
# 13.2 Ablation / ablation study
# =========================================================
ABLATIONS = {"no_fir": ("dann", {"fir_p": 0.0}, "dann"), "no_aug": ("dann", {"augment": False}, "dann"),
             "no_balance": ("dann", {"balanced": False}, "dann"), "cdan_noent": ("cdan", {}, "cdan_e")}
if cfg.run_ablation:
    todo = {k: v for k, v in ABLATIONS.items() if k in cfg.ablation_variants and v[2] in cfg.methods}
    jobs = []
    for name, (m, hp, _) in todo.items():
        jobs += main_jobs(variant=f"abl_{name}", methods=[m], seeds=SW_S, tasks=[SW_T], hp=hp)
    run_jobs(jobs, "ablation")
    ab_rows = []
    for name, (m, hp, refm) in todo.items():
        S = method_summary(SW_T, m, f"abl_{name}", seeds=SW_S)
        R = method_summary(SW_T, refm, "main", seeds=SW_S)
        if S is None or R is None:
            continue
        common = S["g"].index.intersection(R["g"].index)
        d, lo, hi, p = paired_delta(S["g"].loc[common, "y"], S["g"].loc[common, "p"], R["g"].loc[common, "p"], seed=5)
        ab_rows.append(dict(variant=name, method=mlabel(m), reference=mlabel(refm), AUC=S["ens"]["auc"],
                            AUC_reference=R["ens"]["auc"], dAUC=d, dAUC_lo=lo, dAUC_hi=hi, p_boot=p))
    if ab_rows:
        ABL = pd.DataFrame(ab_rows); ABL["p_holm"] = holm(ABL.p_boot.values)
        save_table(ABL, "ablation",
                   f"Ablation on {TASKS[SW_T]['name']} (seeds {list(SW_S)}): target AUC of each variant, its reference "
                   "configuration, and paired ΔAUC (variant − reference) with Holm correction.", 3)
        fig, ax = plt.subplots(figsize=(7, 0.5 * len(ABL) + 1.4))
        for i, r in ABL.iterrows():
            ax.errorbar(r.dAUC, i, xerr=[[r.dAUC - r.dAUC_lo], [r.dAUC_hi - r.dAUC]], fmt="o", capsize=3,
                        color=METHOD_COLOR.get("dann" if r.variant != "cdan_noent" else "cdan_e"))
        ax.axvline(0, color="k", lw=1)
        ax.set_yticks(range(len(ABL))); ax.set_yticklabels([f"{v} (vs {rf})" for v, rf in zip(ABL.variant, ABL.reference)])
        ax.invert_yaxis(); ax.set_xlabel("ΔAUC (variant − reference), paired bootstrap")
        savefig(fig, "ablation", "Effect of removing individual components (negative = the component helps).")
else:
    print("Bỏ qua ablation (cfg.run_ablation=False).")

## 14. (Tuỳ chọn) Cần bao nhiêu bản ghi đích có nhãn? / Semi-supervised curve (RQ5)

Bật `cfg.run_ssda=True`. Với *n* bản ghi đích **có nhãn mỗi lớp** (chọn ngẫu nhiên trong `tgt_train` của fold): so sánh
"nguồn + *n* nhãn đích" (không căn chỉnh) với "DANN + *n* nhãn đích". Điểm *n* = 0 lấy từ lượt chính (source-only, DANN),
"all" = oracle. Đường cong này cho biết chi phí gán nhãn hiện trường cần thiết để thay thế UDA.

In [ ]:
# =========================================================
# 14.1 Đường cong bán giám sát / semi-supervised learning curve
# =========================================================
if cfg.run_ssda and "dann" in cfg.methods:
    jobs = []
    for n in cfg.ssda_shots:
        jobs += main_jobs(variant="ssda", methods=["oracle", "dann"], seeds=SW_S, tasks=[SW_T], shots=int(n))
    run_jobs(jobs, "semi-supervised")
    xs = [0] + [int(n) for n in cfg.ssda_shots]
    curve = []
    for fam, m0 in [("Source + target labels (no adaptation)", "source_only"), ("DANN + target labels", "dann")]:
        for n in xs:
            S = (method_summary(SW_T, m0, "main", seeds=SW_S) if n == 0 else
                 method_summary(SW_T, "oracle" if m0 == "source_only" else "dann", "ssda", seeds=SW_S, shots=n))
            if S is not None:
                lo, hi = boot_auc_ci(S["g"].y, S["g"].p, seed=n)
                curve.append(dict(family=fam, shots_per_class=n, AUC=S["ens"]["auc"], lo=lo, hi=hi))
    CUR = pd.DataFrame(curve)
    save_table(CUR, "ssda_curve", f"Target AUC versus number of labelled target recordings per class "
                                  f"({TASKS[SW_T]['name']}, seeds {list(SW_S)}).", 3)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for (fam, d), c in zip(CUR.groupby("family", sort=False), ["#6E7B8B", METHOD_COLOR["dann"]]):
        ax.errorbar(d.shots_per_class, d.AUC, yerr=[d.AUC - d.lo, d.hi - d.AUC], fmt="o-", capsize=3, color=c, label=fam)
    orc = method_summary(SW_T, "oracle", "main", seeds=SW_S)
    if orc is not None:
        ax.axhline(orc["ens"]["auc"], color=METHOD_COLOR["oracle"], ls="-.", label="all target labels (oracle)")
    ax.set_xlabel("labelled target recordings per class"); ax.set_ylabel("target record-level AUC"); ax.legend(fontsize=8)
    savefig(fig, "ssda_curve", "Semi-supervised adaptation: target AUC as labelled target recordings are added.")
else:
    print("Bỏ qua đường cong bán giám sát (đặt cfg.run_ssda=True để chạy).")

## 15. Mô hình dựa vào dải tần nào trên miền đích? / Band-occlusion sensitivity

Che lần lượt từng nhóm dải Mel (đặt về mức trung bình sau chuẩn hoá) trên **frame kiểm thử đích** (fold 0) và đo mức giảm AUC
bản ghi. So sánh source-only với các mô hình đã thích ứng cho thấy thích ứng **dịch chuyển sự chú ý** sang dải nào — đối chiếu
với bảng AUC theo dải trong từng miền (mục 3.5).

In [ ]:
# =========================================================
# 15.1 Occlusion theo nhóm dải Mel / mel-band occlusion
# =========================================================
if cfg.run_band_occlusion and cfg.backbone == "cnn2d" and KEPT:
    G = 8
    edges = np.linspace(0, SPEC_SHAPE[1], G + 1).astype(int)
    mel_hz = librosa.mel_frequencies(n_mels=SPEC_SHAPE[1] + 2, fmin=cfg.f_min, fmax=F_MAX_EFF, htk=True)[1:-1]
    labels_ = [f"{mel_hz[a]:.0f}–{mel_hz[b - 1]:.0f}" for a, b in zip(edges[:-1], edges[1:])]
    occ_rows = []
    for m, net in KEPT.items():
        base, occ = [], [[] for _ in range(G)]
        with torch.no_grad():
            for s in range(0, len(tgt_te0), 256):
                S_ = net.spec(fetch(tgt_te0[s:s + 256], DEVICE))
                base.append(torch.sigmoid(net.head(net.trunk_from_spec(S_))[0].float()).cpu().numpy())
                for gi, (a, b) in enumerate(zip(edges[:-1], edges[1:])):
                    S2 = S_.clone(); S2[:, :, a:b, :] = 0.0
                    occ[gi].append(torch.sigmoid(net.head(net.trunk_from_spec(S2))[0].float()).cpu().numpy())
        gb = rec_probs_from(tgt_te0, np.concatenate(base)); a0 = safe_auc(gb.y, gb.p)
        for gi in range(G):
            go = rec_probs_from(tgt_te0, np.concatenate(occ[gi]))
            occ_rows.append(dict(method=mlabel(m), band_hz=labels_[gi], auc_base=a0,
                                 auc_drop=a0 - safe_auc(go.y, go.p)))
    OCC = pd.DataFrame(occ_rows)
    save_table(OCC.pivot(index="band_hz", columns="method", values="auc_drop").reindex(labels_).reset_index(),
               "band_occlusion", f"Drop in target recording-level AUC when each mel-band group is occluded "
                                 f"({TASKS[FS_T]['name']}, fold-0 test recordings).", 3)
    fig, ax = plt.subplots(figsize=(12, 3.8))
    ms_ = OCC.method.unique(); w = .8 / len(ms_)
    for k, m in enumerate(ms_):
        d = OCC[OCC.method == m].set_index("band_hz").reindex(labels_)
        key = [kk for kk, vv in METHOD_LABEL.items() if vv == m]
        ax.bar(np.arange(G) + (k - (len(ms_) - 1) / 2) * w, d.auc_drop, w,
               color=METHOD_COLOR.get(key[0] if key else "", "k"), label=m)
    ax.axhline(0, color="k", lw=.8)
    ax.set_xticks(range(G)); ax.set_xticklabels(labels_, rotation=20); ax.set_xlabel("occluded band (Hz)")
    ax.set_ylabel("AUC drop"); ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    savefig(fig, "band_occlusion",
            "Band-occlusion sensitivity on target test recordings: larger drops mean the model relies more on that band.")
else:
    print("Bỏ qua occlusion (backbone không phải cnn2d hoặc không có mô hình đã giữ).")

## 16. Xuất tư liệu cho bài báo / Export artefacts for the manuscript

- `tables/`: mỗi bảng ở **CSV + LaTeX (booktabs) + Markdown**, đánh số theo thứ tự xuất hiện; `main_results_numeric.csv` giữ số thô.
- `figures/`: mỗi hình **PNG 300 dpi + PDF vector**; `figure_index.csv` chứa chú thích.
- `manuscript_draft.md`: đoạn **Methods/Results** tiếng Anh sinh tự động từ số liệu đo được (chỉ khẳng định "cải thiện có ý nghĩa"
  khi CI của ΔAUC không chứa 0 **và** p Holm < 0.05) + tóm tắt tiếng Việt.
- `ada_summary.json`, `checkpoints/`, `ada_paper_artifacts.zip`.

In [ ]:
# =========================================================
# 16.1 Bảng siêu tham số / hyper-parameter table
# =========================================================
hp_rows = [
    ("Data", "Domains", ", ".join(DOMAINS_ALL)),
    ("Data", "Independent recordings", f"{len(records)} ({int(records.label.sum())} leak)"),
    ("Data", "Frame / hop", f"{cfg.frame_sec:g} s / {cfg.hop_sec:g} s at {cfg.target_sr} Hz"),
    ("Pre-processing", "Filters", f"DC removal, {cfg.highpass_hz:g} Hz high-pass, "
                                  f"{'50 Hz notch ×' + str(cfg.mains_harmonics) if cfg.notch_mains else 'no notch'}, per-file peak norm"),
    ("Front-end", "log-Mel", f"{cfg.n_mels} bands, n_fft {cfg.n_fft}, hop {cfg.hop_length}, {cfg.f_min:g}–{F_MAX_EFF:.0f} Hz"
                             + (" (bandwidth-harmonised)" if HARMONISED else "")),
    ("Front-end", "SpecAugment", f"2 freq masks ≤{cfg.specaug_freq_mask}, 2 time masks ≤{cfg.specaug_time_mask}"),
    ("Network", "G_f widths / bottleneck", f"{list(cfg.widths)} / {cfg.bottleneck_dim}"),
    ("Network", "Discriminator", f"MLP {cfg.disc_hidden}-{cfg.disc_hidden}-1, dropout {cfg.disc_dropout}"),
    ("Adaptation", "λ schedule", f"λ_max(2/(1+exp(−{cfg.lambda_gamma:g}p))−1), λ_max = {cfg.lambda_max:g}"),
    ("Adaptation", "CORAL / MMD weight", f"{cfg.coral_weight:g} / {cfg.mmd_weight:g} (same ramp)"),
    ("Optimisation", "Optimiser", f"AdamW, lr {cfg.lr:g} (disc ×{cfg.disc_lr_mult:g}), wd {cfg.weight_decay:g}"),
    ("Optimisation", "Schedule", f"{cfg.epochs} epochs × {cfg.iters_per_epoch} iterations, warm-up {cfg.warmup_frac:.0%} + cosine"),
    ("Optimisation", "Batch", f"{cfg.batch_size} source (class-balanced={cfg.class_balanced_source}) + {cfg.batch_size} target"),
    ("Optimisation", "Regularisation", f"label smoothing {cfg.label_smoothing}, grad-clip {cfg.grad_clip}, AMP={cfg.amp}"),
    ("Augmentation", "Waveform", f"random FIR p={cfg.aug_fir_p}, noise SNR {cfg.aug_noise_snr_db} dB p={cfg.aug_noise_p}, "
                                 f"circular shift p={cfg.aug_shift_p}"),
    ("Protocol", "Target", f"{cfg.target_protocol}, {cfg.tgt_folds}-fold over target recordings"),
    ("Protocol", "Source validation", f"{cfg.src_val_frac:.0%} of source recordings"),
    ("Protocol", "Repetitions", f"seeds {list(cfg.seeds)}"),
    ("Protocol", "Checkpoint rule", f"{cfg.select_rule} (from epoch {SELECT_FROM_EPOCH})"),
    ("Statistics", "Record aggregation / threshold", f"{cfg.record_agg} / {cfg.threshold}"),
    ("Statistics", "CIs and tests", f"bootstrap over recordings (B = {cfg.bootstrap_n}), paired ΔAUC, Holm"),
]
save_table(pd.DataFrame(hp_rows, columns=["group", "parameter", "value"]), "hyperparameters",
           "Experimental settings and hyper-parameters (all fixed before looking at target-domain results).", 3)

In [ ]:
# =========================================================
# 16.2 Văn bản nháp Methods/Results + tóm tắt + đóng gói / draft text, summary JSON, archive
# =========================================================
def _row(task, m):
    d = MAIN[(MAIN.task == task) & (MAIN.method_key == m)]
    return None if d.empty else d.iloc[0]

def _sig(r):
    return (r is not None and np.isfinite(r.dAUC_lo) and (r.dAUC_lo > 0 or r.dAUC_hi < 0)
            and np.isfinite(r.p_holm) and r.p_holm < .05)

L = ["# Draft text (auto-generated from measured results — verify before use)\n", "## Methods (draft)\n"]
L.append(
    f"Recordings from {len(DOMAINS_ALL)} acquisition domains ({', '.join(DOMAINS_ALL)}) of the tapdulieuMix_16k corpus were "
    f"used ({len(records)} independent recordings, {records.duration_s.sum() / 60:.1f} min; recording identifiers were "
    f"qualified by domain). Signals were resampled to {cfg.target_sr} Hz, high-pass filtered at {cfg.highpass_hz:g} Hz, "
    f"notch-filtered at 50 Hz and harmonics, peak-normalised per file and split into {cfg.frame_sec:g}-s frames with "
    f"{cfg.hop_sec:g}-s hop. "
    + (f"Because the domains differed in effective bandwidth, all spectral front-ends were limited to the common band "
       f"0–{F_MAX_EFF:.0f} Hz. " if HARMONISED else "")
    + f"A log-Mel CNN feature extractor with a {cfg.bottleneck_dim}-d bottleneck was shared by all methods. We compared "
      f"source-only training, AdaBN, Deep CORAL, DANN (gradient reversal, λ_p ramped to λ_max = {cfg.lambda_max:g}) and "
      f"CDAN+E, with a supervised source+target oracle as an upper reference. Each transfer task was evaluated with an "
      f"inductive {cfg.tgt_folds}-fold split of the target recordings: unlabelled target data used for adaptation never "
      f"included the recordings being tested. Training used AdamW ({cfg.epochs}×{cfg.iters_per_epoch} iterations, "
      f"batch {cfg.batch_size}+{cfg.batch_size}); the checkpoint was chosen by the pre-registered '{cfg.select_rule}' rule "
      f"without target labels. Experiments were repeated with {len(cfg.seeds)} seeds (new partitions and initialisations) "
      f"and out-of-fold probabilities were averaged across seeds. The primary metric is the recording-level AUC "
      f"(mean frame probability); 95% CIs were obtained by bootstrap over target recordings ({cfg.bootstrap_n} resamples) "
      f"and methods were compared with source-only by paired bootstrap ΔAUC with Holm correction.\n")
L.append("\n## Results (draft)\n")
VN = []
for T in TASKS:
    so, dn, ce, orc = (_row(T["name"], m) for m in ("source_only", "dann", "cdan_e", "oracle"))
    s = f"**{T['name']}.** "
    if so is not None:
        s += f"Source-only reached a target AUC of {_ci(so.AUC, so.AUC_lo, so.AUC_hi)}. "
    for r in (dn, ce):
        if r is None:
            continue
        verdict = ("significantly improved" if _sig(r) and r.dAUC > 0 else
                   "significantly reduced" if _sig(r) else "did not significantly change")
        s += (f"{r.method} {verdict} the target AUC ({r.AUC:.3f}; ΔAUC = {_ci(r.dAUC, r.dAUC_lo, r.dAUC_hi)}, "
              f"Holm p = {r.p_holm:.3f}" + (f", {r.gap_closed_pct:.0f}% of the source–oracle gap closed"
                                             if np.isfinite(r.gap_closed_pct) else "") + "). ")
        VN.append(f"- {T['name']}: {r.method} AUC {r.AUC:.3f} (ΔAUC {r.dAUC:+.3f} [{r.dAUC_lo:+.3f}, {r.dAUC_hi:+.3f}], "
                  f"p Holm {r.p_holm:.3f}) → " + ("CẢI THIỆN có ý nghĩa" if _sig(r) and r.dAUC > 0 else
                                                  "GIẢM có ý nghĩa" if _sig(r) else "KHÔNG khác biệt có ý nghĩa"))
    if orc is not None:
        s += f"The supervised oracle reached {_ci(orc.AUC, orc.AUC_lo, orc.AUC_hi)}."
    L.append(s + "\n")
if "FS_TAB" in globals() and len(FS_TAB):
    pads = {r.representation: r.PAD for r in FS_TAB.itertuples()}
    L.append(f"\nOn {TASKS[FS_T]['name']}, the proxy A-distance between source and target features was "
             + ", ".join(f"{k}: {v:.2f}" for k, v in pads.items()) + ".\n")
if "SEL" in globals() and len(SEL):
    o = SEL.optimism.dropna()
    if len(o):
        L.append(f"\nSelecting checkpoints with target labels would have inflated target AUC by {o.mean():+.3f} on average "
                 f"(max {o.max():+.3f}) relative to the label-free '{cfg.select_rule}' rule.\n")
L.append("\n## Tóm tắt tiếng Việt\n" + "\n".join(VN) + "\n")
(OUT / "manuscript_draft.md").write_text("\n".join(L), encoding="utf-8")
display(Markdown("\n".join(L)))

pd.DataFrame([dict(figure=k, caption=v) for k, v in FIG_INDEX.items()]).to_csv(OUT / "figure_index.csv", index=False)
pd.DataFrame([dict(table=k, caption=v) for k, v in TAB_INDEX.items()]).to_csv(OUT / "table_index.csv", index=False)
summary = dict(version="ada-v1", run_fingerprint=RUN_FP, synthetic=bool(SYNTHETIC), domains=DOMAINS_ALL,
               tasks=[T["name"] for T in TASKS], f_max_eff=F_MAX_EFF, harmonised=bool(HARMONISED),
               n_records=int(len(records)), n_frames=int(N_FRAMES), config=_cfg_print,
               main_results=MAIN.drop(columns=[], errors="ignore").to_dict(orient="records"),
               runtime_min=(time.time() - T_START) / 60,
               software=dict(python=sys.version.split()[0], torch=torch.__version__,
                             torchaudio=torchaudio.__version__, numpy=np.__version__),
               gpus=[torch.cuda.get_device_properties(i).name for i in range(N_GPU)])
json.dump(summary, open(OUT / "ada_summary.json", "w"), indent=1, ensure_ascii=False, default=float)

with zipfile.ZipFile(OUT / "ada_paper_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in list(FIG_DIR.glob("*")) + list(TAB_DIR.glob("*")):
        z.write(p, p.relative_to(OUT))
    for name in ("manuscript_draft.md", "figure_index.csv", "table_index.csv", "ada_summary.json",
                 "manifest_files.csv", "manifest_records.csv"):
        if (OUT / name).exists():
            z.write(OUT / name, name)
print(f"Đã xuất {len(FIG_INDEX)} hình, {len(TAB_INDEX)} bảng -> {OUT / 'ada_paper_artifacts.zip'}")
print(f"Tổng thời gian chạy: {(time.time() - T_START) / 60:.1f} phút")
if SYNTHETIC:
    print("!! ĐANG Ở SYNTHETIC DEMO MODE — mọi số liệu trên là của dữ liệu mô phỏng, KHÔNG dùng cho bài báo.")

## 17. Diễn giải, hạn chế và danh mục kiểm tra trước khi viết bài / Interpretation, limitations, checklist

**Đọc kết quả theo thứ tự:**

1. **Mục 3 có cho thấy dịch chuyển miền thật không?** PAD đầu vào cao và AUC chéo miền (Bảng ma trận A→B) thấp hơn
   rõ rệt so với đường chéo → có động cơ cho thích ứng. Nếu AUC chéo miền ≈ trong miền, UDA không có gì để cải thiện.
2. **Giả thiết UDA có đứng không?** Bảng AUC theo dải trong từng miền: dải "FLIP" có ý nghĩa ⇒ dịch chuyển có điều kiện;
   tỉ lệ lớp khác nhau nhiều ⇒ dịch chuyển nhãn. Cả hai làm DANN (căn chỉnh biên) kém tin cậy; CDAN+E và PAD theo lớp
   (mục 11) cho biết căn chỉnh có điều kiện có xảy ra không.
3. **Bảng kết quả chính**: chỉ viết "DANN cải thiện" khi CI của ΔAUC không chứa 0 **và** p Holm < 0.05; báo cáo cả
   *gap closed* và AUC nguồn-val (DANN không được phá hiệu năng nguồn).
4. **Mục 10–11**: độ chính xác bộ phân biệt → ~0.5, PAD giảm, silhouette lớp đích không giảm = thích ứng đúng nghĩa.
   PAD giảm nhưng silhouette giảm = **căn chỉnh sai lớp (negative transfer)**.
5. **Mục 12**: báo cáo kết quả theo quy tắc **không dùng nhãn đích**; nêu mức lạc quan của `oracle` như một phát hiện về phương pháp luận.

**Hạn chế phải nêu trong bài báo:**

- Số **bản ghi** đích nhỏ ⇒ CI rộng; mọi kết luận ở mức bản ghi, không phải frame.
- "Miền" ở đây là **nhãn thao tác** (thư mục) — có thể gộp nhiều yếu tố (cảm biến, điểm khảo sát, áp suất, chuỗi thu).
  Nếu có metadata chi tiết, nên tách miền theo từng yếu tố (`cfg.domain_source="csv"`).
- Quy nạp K-fold dùng ít dữ liệu đích không nhãn hơn kịch bản *transductive* thường gặp trong tài liệu UDA → kết quả bảo thủ hơn.
- Siêu tham số (λ_max, lịch, kiến trúc) cố định trước; quét λ (mục 13) chỉ là phân tích độ nhạy, không dùng để chọn.

| Kiểm tra trước khi nộp | Mục |
|---|---|
| Không còn cặp trùng nội dung giữa nguồn và đích | 3.6 |
| Băng thông các miền đã hoà hợp (hoặc nêu lý do không cần) | 3.2 |
| 5 assertion phân hoạch đạt cho mọi seed | 6.2 |
| Mọi bản ghi đích có đúng một dự đoán OOF mỗi seed | 9.2 (dòng "phủ") |
| Kết quả chính dùng quy tắc chọn không nhãn đích | 9.2, 12 |
| SYNTHETIC = False | 16.2 |